ESTUDIO GRAFOS GAME OF THRONES

In [129]:
import csv
def parse_grafo(ruta):
    g=[]
    with open(ruta , encoding="utf-8") as f:
        lector= csv.reader(f)
        next(lector) 
        for linea in lector:
            relacion=tuple()
            relacion=(linea[0], linea[1], int(linea[2]))
            g.append(relacion)
        return Graph(g, weighted=True)

In [130]:
base = "../data/processed/got/"
g_ep1= parse_grafo(base+"got_s1_edges_clean.csv")
g_ep2= parse_grafo(base+"got_s2_edges_clean.csv")
g_ep3= parse_grafo(base+"got_s3_edges_clean.csv")
g_ep4= parse_grafo(base+"got_s4_edges_clean.csv")
g_ep5= parse_grafo(base+"got_s5_edges_clean.csv")
g_ep6= parse_grafo(base+"got_s6_edges_clean.csv")
g_ep7= parse_grafo(base+"got_s7_edges_clean.csv")
g_ep8= parse_grafo(base+"got_s8_edges_clean.csv")
g_general= parse_grafo(base+"got_full_interactions_edges_clean.csv")

In [133]:
import networkx as nx

G_ep1 = nx.Graph()
for a, b, peso in g_ep1.edges():
    G_ep1.add_edge(a, b, weight=peso)
    
G_ep2 = nx.Graph()
for a, b, peso in g_ep2.edges():
    G_ep2.add_edge(a, b, weight=peso)
    
G_ep3 = nx.Graph()
for a, b, peso in g_ep3.edges():
    G_ep3.add_edge(a, b, weight=peso)
    
G_ep4 = nx.Graph()
for a, b, peso in g_ep4.edges():
    G_ep4.add_edge(a, b, weight=peso)
    
G_ep5 = nx.Graph()
for a, b, peso in g_ep5.edges():
    G_ep5.add_edge(a, b, weight=peso)
        
G_ep6 = nx.Graph()
for a, b, peso in g_ep6.edges():
    G_ep6.add_edge(a, b, weight=peso)
    
G_ep7 = nx.Graph()
for a, b, peso in g_ep7.edges():
    G_ep7.add_edge(a, b, weight=peso)

G_ep8 = nx.Graph()
for a, b, peso in g_ep8.edges():
    G_ep8.add_edge(a, b, weight=peso)

G_general = nx.Graph()
for a, b, peso in g_general.edges():
    G_general.add_edge(a, b, weight=peso)

In [107]:
#AUXILIAR
def calcular_caminos(G):
    if nx.is_connected(G):
        print("El grafo es conexo (1 componente).")
        print(f"Camino medio: {nx.average_shortest_path_length(G)}")
        print(f"Diámetro: {nx.diameter(G)}")
    else:
        componentes = sorted(nx.connected_components(G), key=len, reverse=True)
        print(f"El grafo NO es conexo ({len(componentes)} componentes).")
        
        for i, comp in enumerate(componentes, start=1):
            subG = G.subgraph(comp)
            if i == 1:
                print(f"\nComponente 1 - Principal ({len(comp)} nodos):")
            else:
                print(f"\nComponente {i} ({len(comp)} nodos): {list(comp)}")
                
            if len(comp) > 1:
                print(f"  Camino medio: {nx.average_shortest_path_length(subG)}")
                print(f"  Diámetro: {nx.diameter(subG)}")
            else:
                print("  Camino medio: 0")
                print("  Diámetro: 0")

<h2>Metodología</h2>

<h3>Construcción de los grafos</h3>
<p>Los datos proceden del dataset de interacciones entre personajes de <i>Juego de Tronos</i> (basado en los guiones de la serie de HBO). Cada personaje es un nodo y se crea una arista entre dos personajes cuando ambos coinciden o interactúan en la misma escena. El peso de la arista refleja la frecuencia o intensidad de dichas interacciones acumuladas. Se ha construido un grafo no dirigido y ponderado para cada temporada (1 a 8) y otro con la red completa de la serie.</p>

<h3>Métricas empleadas</h3>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Métrica</th><th>Qué mide</th><th>Interpretación en la red</th></tr>
  <tr><td>Grado</td><td>Número de personajes distintos con los que interactúa</td><td>Cantidad de contactos directos</td></tr>
  <tr><td>Grado ponderado</td><td>Suma de los pesos de sus aristas</td><td>Volumen total de interacciones acumuladas con otros personajes</td></tr>
  <tr><td>Intermediación (betweenness)</td><td>Proporción de caminos mínimos entre pares de personajes que pasan por el nodo</td><td>Personajes que actúan de puente entre núcleos narrativos o regiones</td></tr>
  <tr><td>Autovector (eigenvector)</td><td>Importancia de un nodo en función de la importancia de sus vecinos</td><td>Estar conectado con los personajes más influyentes, no solo con muchos</td></tr>
  <tr><td>Cercanía (closeness)</td><td>Inverso de la distancia media al resto de nodos</td><td>Rapidez con la que un personaje alcanza al resto de la red</td></tr>
</table>

<p>La intermediación y la cercanía se calculan sin pesos, de modo que reflejan únicamente la topología estructural de la red. La razón es que en este grafo un peso alto indica mayor cercanía o relación, mientras que estos algoritmos interpretan el peso como una distancia. El grado ponderado y el autovector sí utilizan los pesos.</p>

<p>Para describir la estructura global se calculan el número de nodos y aristas, la densidad, las componentes conexas, el camino medio y el diámetro. En aquellas temporadas en las que la red no es conexa (debido a tramas geográficamente aisladas), el camino medio y el diámetro se calculan sobre la componente conexa principal. Las comunidades se detectan con el algoritmo de maximización voraz de la modularidad (<i>greedy modularity</i>), empleando los pesos de las aristas.</p>

<h3>Limitaciones del método</h3>
<ul>
  <li><b>Sesgo de reunión:</b> al registrar interacciones por escena, cuando varios personajes coinciden en consejos, banquetes o batallas todos quedan conectados entre sí. Esto infla el grado de personajes secundarios que asisten a escenas grupales, por lo que el grado se contrasta siempre con el grado ponderado.</li>
  <li><b>Fragmentación geográfica:</b> debido a la estructura coral de la serie con múltiples líneas argumentales separadas (Desembarco del Rey, el Norte, Essos), algunas temporadas presentan componentes inconexas hasta que los personajes convergen.</li>
  <li><b>Menciones y recuerdos:</b> los nodos se basan en la presencia e interacción directa en escena, sin incluir alusiones indirectas entre personajes ausentes.</li>
</ul>

<h3>Análisis grafo temporada 1</h3>

<p>Grado</p>

In [108]:
grado = dict(G_ep1.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('NED', 57), ('TYRION', 41), ('CATELYN', 36), ('ROBERT', 36), ('ROBB', 30), ('CERSEI', 29), ('ARYA', 28), ('JOFFREY', 27), ('SANSA', 26), ('JON', 26)]


<p>Grado ponderado.</p>

In [109]:
grado_pond = dict(G_ep1.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('NED', 1290), ('TYRION', 709), ('CATELYN', 584), ('ROBERT', 563), ('JON', 535), ('DAENERYS', 535), ('CERSEI', 444), ('ROBB', 424), ('SANSA', 422), ('LITTLEFINGER', 383)]


<p>Intermediación (betweenness)</p>

In [110]:
inter = nx.betweenness_centrality(G_ep1)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('NED', 0.3032802081478769), ('TYRION', 0.16302743524270863), ('CATELYN', 0.11828405295282025), ('ROBERT', 0.11039475684723858), ('DAENERYS', 0.10143128748178008), ('JON', 0.08533639644915646), ('ROBB', 0.08000074140585357), ('ARYA', 0.0660059483301368), ('PYP', 0.06091067421463666), ('JORAH', 0.053910440465872796)]


<p>Autovector (eigenvector)</p>

In [111]:
auto = nx.eigenvector_centrality(G_ep1, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('NED', 0.5624687154111963), ('ROBERT', 0.3851054478657294), ('CERSEI', 0.28868934342650576), ('LITTLEFINGER', 0.24609956461674143), ('CATELYN', 0.23042540131059233), ('VARYS', 0.2154278104941654), ('SANSA', 0.20389677480100651), ('ARYA', 0.19905531335746823), ('JOFFREY', 0.1940173446682978), ('JAIME', 0.1724297188883803)]


<p>Cercanía (closeness)</p>

In [112]:
cerc = nx.closeness_centrality(G_ep1)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('NED', 0.628140703517588), ('ROBERT', 0.5530973451327433), ('CATELYN', 0.5506607929515418), ('TYRION', 0.5434782608695652), ('JON', 0.5186721991701245), ('ROBB', 0.5122950819672131), ('JOFFREY', 0.5102040816326531), ('CERSEI', 0.508130081300813), ('ARYA', 0.5020080321285141), ('JAIME', 0.49800796812749004)]


<p>Comunidades</p>

In [113]:
comunidades = nx.community.greedy_modularity_communities(G_ep1, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

41
['PYP', 'BOWEN_MARSH', 'BORCAS', 'GRENN', 'LUKE', 'ALLISER_THORNE', 'RAST', 'OTHOR', 'JEOR', 'DAREON', 'OTHELL_YARWYCK', 'JAREMY_RYKKER']
['AEGON', 'MAESTER_AEMON', 'BRANDON_STARK', 'RHAEGAR', 'SEPTA_MORDANE', 'AERYS', 'RICKARD_STARK']
['MERYN_TRANT', 'BAELOR', 'MARILLION', 'YOREN', 'ILYN_PAYNE', 'HOUND']
['DOREAH', 'WINE_MERCHANT', 'IROGENIA', 'IRRI', 'RAKHARO', 'JHIQUI']
['ARYA', 'TOBHO_MOTT', 'STABLE_BOY', 'HOT_PIE', 'LOMMY_GREENHANDS', 'GENDRY']
['ADDAM_MARBRAND', 'KEVAN', 'SHAGGA', 'LEO_LEFFORD', 'TYWIN']
['MORD', 'LYSA', 'BRONN', 'ROBIN', 'VARDIS_EGEN']
['LORAS', 'HUGH_OF_THE_VALE', 'MACE', 'MOUNTAIN', 'JON_ARRYN']
['TOMARD', 'RENLY', 'STANNIS', 'GREATJON_UMBER', 'RICKARD_KARSTARK']
['ROOSE_BOLTON', 'HIGH_SEPTON', 'VARLY', 'NED', 'STEFFON']
['HOSTER', 'STEVRON_FREY', 'WALDER', 'JOYEUSE', 'RYGER_RIVERS']
['MAGO', 'COHOLLO', 'DROGO', 'ILLYRIO']
['GARED', 'WAYMAR_ROYCE', 'WILL', 'ROYCE']
['BALON', 'OLD_NAN', 'THEON']
['OSHA', 'HODOR', 'RICKON']
['ROBB', 'JONOS_BRACKEN', 'GALBART_

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [114]:
print(
    f"nodos: {G_ep1.number_of_nodes()} aristas: {G_ep1.number_of_edges()}"
    f" densidad: {nx.density(G_ep1)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep1)}")

nodos: 126 aristas: 549 densidad: 0.06971428571428571
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [115]:
print(nx.average_shortest_path_length(G_ep1))
print(nx.diameter(G_ep1))

2.6446984126984128
6


<h3>Resultados de la temporada 1</h3>

<p><b>Estructura.</b> La red tiene 126 personajes y 549 aristas, con una densidad de 0,070. Es un grafo conexo (1 sola componente), con un camino medio de 2,64 y un diámetro de 6. Pese a la dispersión geográfica entre Invernalia, Desembarco del Rey, el Muro y Essos, toda la red queda conectada en un máximo de 6 pasos.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Ned (57)</td><td>Ned (1290)</td><td>Ned (0,303)</td><td>Ned (0,562)</td><td>Ned (0,628)</td></tr>
  <tr><td>2</td><td>Tyrion (41)</td><td>Tyrion (709)</td><td>Tyrion (0,163)</td><td>Robert (0,385)</td><td>Robert (0,553)</td></tr>
  <tr><td>3</td><td>Catelyn (36)</td><td>Catelyn (584)</td><td>Catelyn (0,118)</td><td>Cersei (0,289)</td><td>Catelyn (0,551)</td></tr>
  <tr><td>4</td><td>Robert (36)</td><td>Robert (563)</td><td>Robert (0,110)</td><td>Littlefinger (0,246)</td><td>Tyrion (0,543)</td></tr>
  <tr><td>5</td><td>Robb (30)</td><td>Jon (535)</td><td>Daenerys (0,101)</td><td>Catelyn (0,230)</td><td>Jon (0,519)</td></tr>
</table>
<p><i>Nota: Catelyn y Robert empatan a 36 contactos en grado, y Jon y Daenerys empatan a 535 interacciones en grado ponderado.</i></p>

<p><b>El protagonista.</b> Ned Stark encabeza con absoluta claridad las cinco métricas. Comparte escena con 57 de los otros 125 personajes (un 45,6 %) y acumula un grado ponderado de 1290, casi duplicando al segundo clasificado, Tyrion (709). Su protagonismo responde a su recorrido narrativo: comienza en Invernalia con su familia y se traslada a Desembarco del Rey como Mano del Rey, articulando el eje central de la trama.</p>

<p><b>Grado frente a grado ponderado.</b> En la parte alta ambos rankings coinciden, pero aparecen contrastes claros. Daenerys es quinta en grado ponderado (535) sin figurar entre los diez primeros por grado: interactúa con un círculo cerrado en Essos, pero acumula un volumen muy alto de escenas con ellos. De forma similar, Littlefinger entra en el top 10 ponderado (383) y es cuarto en autovector (0,246) pese a no destacar en grado bruto, reflejando interacciones muy concentradas con el núcleo de poder de la capital (Ned, Robert, Cersei).</p>

<p><b>Puentes entre grupos.</b> Ned (0,303) y Tyrion (0,163) lideran la intermediación. Tyrion actúa como gran conector interregional al viajar desde Invernalia al Muro, posteriormente al Nido de Águilas con Catelyn y finalmente al campamento Lannister. Destacan también Daenerys (quinta, 0,101) y Jorah (décimo, 0,054), quienes sirven de enlace entre toda la trama de Essos y el resto de Poniente, así como Pyp (noveno, 0,061) dentro de la Guardia de la Noche.</p>

<p><b>Comunidades.</b> El algoritmo detecta 41 comunidades, de las cuales 11 son nodos aislados (como Sansa, Jon, Littlefinger o Daenerys, cuyas conexiones se reparten demasiado entre facciones como para quedar adscritos a una sola). Entre las agrupaciones principales destacan:</p>
<ul>
  <li><b>Guardia de la Noche:</b> Jeor, Alliser Thorne, Pyp, Grenn, Rast, Bowen Marsh y Othell Yarwyck (12 personajes).</li>
  <li><b>Corte de Daenerys en Essos:</b> Irri, Jhiqui, Doreah, Rakharo, Irogenia y Wine Merchant, junto con un subgrupo de Drogo, Illyrio, Mago y Cohollo.</li>
  <li><b>Camino Real y Desembarco del Rey (Arya):</b> Arya, Gendry, Hot Pie, Lommy Greenhands, Tobho Mott y Stable Boy.</li>
  <li><b>Ejército Lannister:</b> Tywin, Kevan, Addam Marbrand, Leo Lefford y Shagga.</li>
  <li><b>El Valle de Arryn:</b> Lysa, Robin, Bronn, Vardis Egen y Mord.</li>
</ul>

<h3>Análisis grafo temporada 2</h3>

<p>Grado</p>

In [116]:
grado = dict(G_ep2.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('JOFFREY', 36), ('TYRION', 33), ('CERSEI', 31), ('ROBB', 28), ('ARYA', 27), ('TYWIN', 25), ('CATELYN', 23), ('STANNIS', 22), ('LITTLEFINGER', 20), ('NED', 20)]


<p>Grado ponderado.</p>

In [117]:
grado_pond = dict(G_ep2.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 960), ('CERSEI', 549), ('ARYA', 481), ('THEON', 447), ('JOFFREY', 425), ('ROBB', 420), ('DAENERYS', 383), ('STANNIS', 334), ('SANSA', 325), ('CATELYN', 323)]


<p>Intermediación (betweenness)</p>

In [118]:
inter = nx.betweenness_centrality(G_ep2)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('ARYA', 0.15104217250781987), ('TYRION', 0.141080374441228), ('ROBB', 0.11704699582237153), ('JON', 0.11087767788061638), ('NED', 0.10431444702553143), ('JOFFREY', 0.10049862632402742), ('THEON', 0.08555810575183231), ('CATELYN', 0.06520519055130625), ('TYWIN', 0.06464758652813397), ('CERSEI', 0.06115998137352085)]


<p>Autovector (eigenvector)</p>

In [119]:
auto = nx.eigenvector_centrality(G_ep2, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.5912917386561733), ('CERSEI', 0.46330747497680214), ('JOFFREY', 0.2967869501132143), ('SANSA', 0.2565737196118361), ('SHAE', 0.24344928985630374), ('VARYS', 0.21974184068445515), ('BRONN', 0.18023210583825824), ('HOUND', 0.1411126725755927), ('LITTLEFINGER', 0.13632312564260912), ('PYCELLE', 0.11958257911961817)]


<p>Cercanía (closeness)</p>

In [120]:
cerc = nx.closeness_centrality(G_ep2)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.47667253521126757), ('JOFFREY', 0.47667253521126757), ('NED', 0.4615056818181818), ('ROBB', 0.45125000000000004), ('CERSEI', 0.4472742290748899), ('CATELYN', 0.443367903930131), ('ARYA', 0.4302171610169491), ('LITTLEFINGER', 0.4284018987341772), ('SANSA', 0.4284018987341772), ('TYWIN', 0.42660189075630256)]


<p>Comunidades</p>

In [121]:
comunidades = nx.community.greedy_modularity_communities(G_ep2, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

38
['FARLEN', 'RICKON', 'RODRIK', 'BLACK_LORREN', 'DAGMER', 'HODOR', 'MAESTER_LUWIN', 'PORTAN', 'BILLY']
['KOVARRO', 'PYATT_PREE', 'QUAITHE', 'DOREAH', 'MALAKKO', 'JORAH', 'IRRI', 'XARO']
['CRASTER', 'RANDYLL', 'GRENN', 'BENJEN', 'SAM', 'GILLY', 'EDDISON_TOLLETT', 'MELESSA']
['MOUNTAIN', 'LOMMY', 'POLLIVER', 'HOT_PIE', 'TICKLER', 'AMORY', 'GENDRY']
['TALISA', 'ALTON', 'ROOSE_BOLTON', 'RAMSAY', 'RICKARD_KARSTARK', 'TORRHEN', 'SEPTON']
['ARYA', 'VISENYA', 'SYRIO_FOREL', 'AEGON', 'RHAENYS']
['MHAEGEN', 'ROS', 'JANOS', 'BARRA', 'DAISY']
['PYCELLE', 'NED', 'AERYS', 'JON_ARRYN', 'ROBERT']
['TRYSTANE', 'MYRCELLA', 'ROBIN', 'HIGH_SEPTON', 'LITTLEFINGER']
['YOREN', 'BITER', 'JAQEN', 'RORGE']
['HAYLENE', 'HOUND', 'MANDON', 'PODRICK']
['MATTHOS', 'CRESSEN', 'MELISANDRE', 'SALLADHOR']
['RAKHARO', 'DAENERYS', 'DROGO', 'RHAEGO']
['REGINALD', 'KEVAN', 'HARREN', 'TYWIN']
['MANCE', 'YGRITTE', 'QHORIN', 'LORD_OF_BONES']
['RENNICK', 'QUENT', 'FREY_DAUGHTER', 'ROBB']
['MARGAERY', 'COLEN', 'LORAS']
['ILYN_

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [122]:
print(
    f"nodos: {G_ep2.number_of_nodes()} aristas: {G_ep2.number_of_edges()}"
    f" densidad: {nx.density(G_ep2)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep2)}")

nodos: 129 aristas: 486 densidad: 0.05886627906976744
componentes conexas: 2


<p>Camino medio y diámetro.</p>

In [123]:
calcular_caminos(G_ep2)

El grafo NO es conexo (2 componentes).

Componente 1 - Principal (115 nodos):
  Camino medio: 2.8573607932875666
  Diámetro: 6

Componente 2 (14 nodos): ['KOVARRO', 'DAENERYS', 'QUAITHE', 'PYATT_PREE', 'DOREAH', 'SPICE_KING', 'MALAKKO', 'JORAH', 'IRRI', 'XARO', 'RHAEGO', 'RAKHARO', 'DROGO', 'SILK_KING']
  Camino medio: 1.5934065934065933
  Diámetro: 2


<h3>Resultados de la temporada 2</h3>

<p><b>Estructura.</b> La red tiene 129 personajes y 486 aristas, con una densidad de 0,059. A diferencia de la primera temporada, el grafo <b>no es conexo</b> y se divide en 2 componentes: una componente principal de 115 personajes (camino medio de 2,86 y diámetro de 6) y una componente aislada de 14 personajes correspondiente a la trama de Daenerys en Essos y Qarth (camino medio de 1,59 y diámetro de 2). Tras la muerte de Robert y Ned, la línea narrativa de Essos queda completamente desconectada de Poniente durante esta temporada.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Joffrey (36)</td><td>Tyrion (960)</td><td>Arya (0,151)</td><td>Tyrion (0,591)</td><td>Tyrion (0,477)</td></tr>
  <tr><td>2</td><td>Tyrion (33)</td><td>Cersei (549)</td><td>Tyrion (0,141)</td><td>Cersei (0,463)</td><td>Joffrey (0,477)</td></tr>
  <tr><td>3</td><td>Cersei (31)</td><td>Arya (481)</td><td>Robb (0,117)</td><td>Joffrey (0,297)</td><td>Ned (0,462)</td></tr>
  <tr><td>4</td><td>Robb (28)</td><td>Theon (447)</td><td>Jon (0,111)</td><td>Sansa (0,257)</td><td>Robb (0,451)</td></tr>
  <tr><td>5</td><td>Arya (27)</td><td>Joffrey (425)</td><td>Ned (0,104)</td><td>Shae (0,243)</td><td>Cersei (0,447)</td></tr>
</table>
<p><i>Nota: Tyrion y Joffrey empatan en cercanía (0,477).</i></p>

<p><b>El cambio de protagonismo.</b> Tras la ejecución de Ned Stark, Tyrion asume el peso narrativo principal como Mano del Rey en funciones: lidera el grado ponderado (960, casi doblando a Cersei con 549), el autovector (0,591) y la cercanía (0,477). Joffrey encabeza el grado (36 contactos) debido a sus apariciones en la corte y la batalla del Aguasnegras, pero su grado ponderado cae al quinto puesto (425), reflejando el sesgo de reunión en escenas ceremoniales.</p>

<p><b>Grado frente a grado ponderado.</b> Theon (cuarto en grado ponderado con 447) y Daenerys (séptima con 383) no aparecen entre los diez primeros en grado: ambos protagonizan tramas con pocos interlocutores (la toma de Invernalia y la estancia en Qarth), pero acumulan un número muy elevado de interacciones con ellos. En sentido opuesto, Tywin (25 contactos) y Littlefinger (20) destacan en grado pero quedan fuera del top 10 ponderado.</p>

<p><b>Puentes entre grupos.</b> Arya lidera la intermediación (0,151) por delante de Tyrion (0,141) y Robb (0,117). Su viaje desde Desembarco del Rey hacia el Norte pasando por Harrenhal (donde coincide con Tywin) la convierte en el principal nexo entre los distintos frentes de la Guerra de los Cinco Reyes. Jon (0,111) ocupa el cuarto puesto al conectar la Guardia de la Noche con los salvajes más allá del Muro.</p>

<p><b>Comunidades.</b> Se detectan 38 comunidades (8 de ellas unipersonales, como Jaime, Catelyn, Stannis o Jon). La componente aislada de Essos se subdivide a su vez en tres grupos (Qarth con Xaro y Pyat Pree; Daenerys y Drogo; y los reyes comerciales de Qarth). En Poniente destacan:</p>
<ul>
  <li><b>Invernalia ocupada:</b> Rodrik, Maester Luwin, Hodor, Bran/Rickon (Osha queda aislada) junto con Dagmer y Black Lorren.</li>
  <li><b>Más allá del Muro (expedición):</b> Jeor (aislado), Sam, Gilly, Craster, Grenn y Eddison Tollett.</li>
  <li><b>Harrenhal y cautivos:</b> Hot Pie, Gendry, Lommy, Polliver, Tickler, Amory y Mountain.</li>
  <li><b>Campamento del Norte:</b> Roose Bolton, Ramsay, Talisa, Rickard Karstark y Alton.</li>
</ul>

<h3>Análisis grafo temporada 3</h3>

<p>Grado</p>

In [50]:
grado = dict(G_ep3.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('ROBB', 31), ('TYWIN', 29), ('TYRION', 25), ('CATELYN', 22), ('JOFFREY', 21), ('CERSEI', 20), ('SANSA', 19), ('JON', 19), ('ROOSE_BOLTON', 18), ('JAIME', 17)]


<p>Grado ponderado.</p>

In [51]:
grado_pond = dict(G_ep3.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 653), ('ROBB', 375), ('TYWIN', 366), ('SANSA', 363), ('CERSEI', 354), ('JAIME', 335), ('JON', 323), ('DAENERYS', 294), ('BRAN', 292), ('ARYA', 288)]


<p>Intermediación (betweenness)</p>

In [52]:
inter = nx.betweenness_centrality(G_ep3)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('ROBB', 0.23811635713320708), ('NED', 0.17455490693263), ('ROBERT', 0.1612072816209622), ('BRAN', 0.14802295374376237), ('JON', 0.13598714263346537), ('TYWIN', 0.12858396569237646), ('CATELYN', 0.12351544108875671), ('SAM', 0.10217883119508576), ('TYRION', 0.09865173199290372), ('STANNIS', 0.0769321133265192)]


<p>Autovector (eigenvector)</p>

In [53]:
auto = nx.eigenvector_centrality(G_ep3, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.5484566281223088), ('TYWIN', 0.38777155970333455), ('CERSEI', 0.37050250143084834), ('SANSA', 0.33332747187071904), ('JOFFREY', 0.2695989856383883), ('OLENNA', 0.2122588628105595), ('MARGAERY', 0.21005943627109946), ('SHAE', 0.20943305557200745), ('BRONN', 0.15743032808227603), ('VARYS', 0.12845402754405225)]


<p>Cercanía (closeness)</p>

In [54]:
cerc = nx.closeness_centrality(G_ep3)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('ROBB', 0.4785992217898833), ('NED', 0.45555555555555555), ('CATELYN', 0.4522058823529412), ('TYWIN', 0.4489051094890511), ('JOFFREY', 0.4241379310344828), ('JAIME', 0.41694915254237286), ('TYRION', 0.41694915254237286), ('BRAN', 0.412751677852349), ('STANNIS', 0.40728476821192056), ('SANSA', 0.40594059405940597)]


<p>Comunidades</p>

In [55]:
comunidades = nx.community.greedy_modularity_communities(G_ep3, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

43
['HOSTER', 'BRYNDEN', 'EDMURE', 'ROSLIN', 'LOTHAR', 'WALDER', 'JOYEUSE', 'BLACK_WALDER', 'RICKARD_KARSTARK', 'FREY_SEPTON', 'MERRY']
['PYP', 'RANDYLL', 'HOBB', 'MAESTER_AEMON', 'SAM', 'LITTLE_SAM', 'GILLY', 'DAREON']
['SHIREEN', 'SELYSE', 'AEGON', 'STANNIS', 'SALLADHOR', 'BERT', 'RYLENE']
['BALON', 'TORTURER', 'YARA', 'RAMSAY', 'THEON', 'VIOLET', 'MYRANDA']
['CRASTER', 'KARL_TANNER', 'GRENN', 'RAST', 'JEOR', 'EDDISON_TOLLETT']
['AERYS', 'BARRISTAN', 'NED', 'RADZAI_MO_ERAZ', 'ROBERT']
['PRENDAHL', 'DAARIO', 'GREY_WORM', 'MERO', 'MISSANDEI']
['STEELSHANKS_WALTON', 'LOCKE', 'ROOSE_BOLTON', 'SELWYN', 'QYBURN']
['ILYN_PAYNE', 'MERYN_TRANT', 'BOROS', 'HOUND']
['HODOR', 'ORELL', 'FARMER', 'RICKON']
['TARYN_MANT', 'BRONN', 'HIGH_SEPTON', 'PYCELLE']
['HOT_PIE', 'ANGUY', 'THOROS']
['KRAZNYS_MO_NAKLOZ', 'JORAH', 'GREIZHEN']
['MATTHOS', 'DAVOS', 'BALERION']
['DESMOND', 'ELDRICK', 'SANSA']
['TORMUND', 'MANCE', 'LORD_OF_BONES']
['OLYVAR', 'LITTLEFINGER', 'ROS']
['LORAS', 'MACE', 'RENLY']
['TALISA

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [56]:
print(
    f"nodos: {G_ep3.number_of_nodes()} aristas: {G_ep3.number_of_edges()}"
    f" densidad: {nx.density(G_ep3)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep3)}")

nodos: 124 aristas: 504 densidad: 0.06608969315499606
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [57]:
print(nx.average_shortest_path_length(G_ep3))
print(nx.diameter(G_ep3))

3.2583267768161552
7


<h3>Resultados de la temporada 3</h3>

<p><b>Estructura.</b> La red cuenta con 124 personajes y 504 aristas, con una densidad de 0,066. Vuelve a ser un grafo conexo (1 sola componente), con un camino medio de 3,26 y un diámetro de 7. La reconexión de la trama de Essos con Poniente se produce gracias a personajes como Barristan Selmy, que vinculan los recuerdos de Robert/Ned con la corte de Daenerys.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Robb (31)</td><td>Tyrion (653)</td><td>Robb (0,238)</td><td>Tyrion (0,548)</td><td>Robb (0,479)</td></tr>
  <tr><td>2</td><td>Tywin (29)</td><td>Robb (375)</td><td>Ned (0,175)</td><td>Tywin (0,388)</td><td>Ned (0,456)</td></tr>
  <tr><td>3</td><td>Tyrion (25)</td><td>Tywin (366)</td><td>Robert (0,161)</td><td>Cersei (0,371)</td><td>Catelyn (0,452)</td></tr>
  <tr><td>4</td><td>Catelyn (22)</td><td>Sansa (363)</td><td>Bran (0,148)</td><td>Sansa (0,333)</td><td>Tywin (0,449)</td></tr>
  <tr><td>5</td><td>Joffrey (21)</td><td>Cersei (354)</td><td>Jon (0,136)</td><td>Joffrey (0,270)</td><td>Joffrey (0,424)</td></tr>
</table>
<p><i>Nota: Sansa y Jon empatan a 19 contactos en grado, y Jaime y Tyrion empatan en cercanía (0,417).</i></p>

<p><b>Protagonismo dividido.</b> La centralidad se reparte entre el frente de la guerra en las Tierras de los Ríos y las intrigas de Desembarco del Rey. Robb encabeza las métricas estructurales: grado (31 contactos), intermediación (0,238) y cercanía (0,479), impulsado por las alianzas y traiciones que culminan en la Boda Roja. Tyrion, en cambio, domina con holgura el grado ponderado (653 frente a los 375 de Robb) y el autovector (0,548), liderando el núcleo más denso de escenas en la capital junto a Tywin, Cersei y Sansa.</p>

<p><b>Grado frente a grado ponderado.</b> Daenerys (294), Bran (292) y Arya (288) entran en el top 10 de grado ponderado sin aparecer entre los diez primeros por grado: protagonizan viajes largos con acompañantes fijos (Jorah/Missandei, Jojen/Meera/Hodor, el Perro/Hermandad), acumulando muchas escenas con pocos personajes. Por el contrario, Roose Bolton (18 contactos) destaca en grado por las escenas corales de guerra y la Boda Roja, pero no entra en el top 10 ponderado.</p>

<p><b>Puentes entre grupos.</b> Tras Robb (0,238), llaman la atención Ned (0,175) y Robert (0,161) en segunda y tercera posición de intermediación. Aunque ambos personajes ya han fallecido, las menciones y vínculos estructurales presentes en las interacciones del dataset actúan como nodos puente que mantienen conectada la red de Essos (a través de Barristan Selmy) con los Stark y Baratheon. Bran (0,148) y Jon (0,136) completan el top 5 al enlazar el Norte y el Muro.</p>

<p><b>Comunidades.</b> El algoritmo identifica 43 comunidades (15 de un solo elemento, incluyendo a Robb, Jaime, Cersei, Jon y Brienne). Entre los bloques principales sobresalen:</p>
<ul>
  <li><b>Boda Roja (Tully y Frey):</b> Walder Frey, Edmure, Brynden, Roslin, Lothar, Black Walder, Hoster y Rickard Karstark.</li>
  <li><b>Expedición de Sam y Gilly:</b> Sam, Gilly, Little Sam, Maester Aemon, Pyp y Dareon.</li>
  <li><b>Cautiverio de Theon:</b> Ramsay, Theon, Myranda, Violet, Torturer, Balon y Yara.</li>
  <li><b>Bahía de los Esclavos:</b> Daario, Grey Worm, Missandei, Mero y Prendahl.</li>
  <li><b>Cautiverio de Jaime:</b> Roose Bolton, Locke, Steelshanks Walton, Qyburn y Selwyn.</li>
</ul>

<h3>Análisis grafo temporada 4</h3>

<p>Grado</p>

In [58]:
grado = dict(G_ep4.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('JOFFREY', 40), ('TYRION', 38), ('CERSEI', 33), ('SANSA', 31), ('JON', 30), ('JAIME', 29), ('TYWIN', 28), ('SAM', 25), ('OBERYN', 25), ('ARYA', 24)]


<p>Grado ponderado.</p>

In [59]:
grado_pond = dict(G_ep4.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 876), ('CERSEI', 619), ('JAIME', 598), ('TYWIN', 586), ('JON', 542), ('SANSA', 468), ('OBERYN', 468), ('JOFFREY', 376), ('DAENERYS', 349), ('SAM', 339)]


<p>Intermediación (betweenness)</p>

In [60]:
inter = nx.betweenness_centrality(G_ep4)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('JOFFREY', 0.20477266691678075), ('NED', 0.19376950827015704), ('STANNIS', 0.17791068164690788), ('JON', 0.1455752734437189), ('VARYS', 0.10827589172571388), ('BARRISTAN', 0.10274916276058586), ('ARYA', 0.09637296710729372), ('DAENERYS', 0.09234936605377579), ('TYRION', 0.09095759740305437), ('JAIME', 0.07662550759871212)]


<p>Autovector (eigenvector)</p>

In [61]:
auto = nx.eigenvector_centrality(G_ep4, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.4840879057502612), ('JAIME', 0.4350452747999645), ('TYWIN', 0.39650782701290505), ('CERSEI', 0.3860562941746878), ('OBERYN', 0.27003279538467223), ('JOFFREY', 0.21328547630646832), ('BRONN', 0.144732741579568), ('PODRICK', 0.14357948882690583), ('SHAE', 0.13548359808696453), ('SANSA', 0.1329653112560276)]


<p>Cercanía (closeness)</p>

In [62]:
cerc = nx.closeness_centrality(G_ep4)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('JOFFREY', 0.45219899062725305), ('JAIME', 0.4275974911371693), ('NED', 0.41998125083701615), ('CERSEI', 0.4095598798485046), ('TYRION', 0.4065335753176043), ('SANSA', 0.40355166645219404), ('TYWIN', 0.40355166645219404), ('STANNIS', 0.3901953465223342), ('ARYA', 0.3776948091051427), ('TOMMEN', 0.3776948091051427)]


<p>Comunidades</p>

In [63]:
comunidades = nx.community.greedy_modularity_communities(G_ep4, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

41
['CRASTER', 'KARL_TANNER', 'LOCKE', 'CRASTERS_BABY', 'MORAG', 'THREE_EYED_RAVEN', 'RAST', 'HODOR', 'JOJEN', 'JEOR', 'WHITE_WALKER', 'MEERA', 'NIGHT_KING', 'SISSY', 'LEAF']
['WALDA', 'BALON', 'MERYN_TRANT', 'RICKON', 'BOLTON_GUARD', 'YARA', 'SYRIO_FOREL', 'ROOSE_BOLTON', 'RAMSAY', 'ADRACK_HUMBLE', 'THEON', 'RALF', 'FIRST_MATE', 'MYRANDA', 'TANSY']
['LORAS', 'BAELOR', 'FOOL', 'RENLY_DWARF', 'HIGH_SEPTON', 'STANNIS_DWARF', 'RENLY', 'JOFFREY_DWARF', 'JOFFREY', 'ROBB_DWARF', 'ENDREW', 'BALON_DWARF']
['SALLY', 'INNKEEPERS_DAUGHTER', 'RORGE', 'FARMER_HAMLET', 'BITER', 'DYING_MAN', 'LOMMY', 'DONNEL_WAYNWOOD', 'HOUND']
['DAENERYS', 'FENNESZ', 'HIZDAHR', 'MASTER_MIGHDAL', 'VISERYS', 'HIZDAHRS_FATHER', 'RHAEGAR', 'DROGO', 'MANSERVANT']
['CATELYN', 'HOSTER', 'LYSA', 'ROBB', 'NED', 'AERYS', 'JON_ARRYN', 'ROBERT']
['SHIREEN', 'SELYSE', 'MELISANDRE', 'MAESTER_AEMON', 'STANNIS', 'EDDISON_TOLLETT', 'AXELL_FLORENT', 'DAVOS']
['GOATHERDS_SON', 'ELDER_MEEREEN_SLAVE', 'GOATHERD', 'BARRISTAN', 'MOSSADOR'

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [64]:
print(
    f"nodos: {G_ep4.number_of_nodes()} aristas: {G_ep4.number_of_edges()}"
    f" densidad: {nx.density(G_ep4)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep4)}")

nodos: 172 aristas: 667 densidad: 0.04535563715490276
componentes conexas: 2


<p>Camino medio y diámetro.</p>

In [125]:
calcular_caminos(G_ep4)

El grafo NO es conexo (2 componentes).

Componente 1 - Principal (169 nodos):
  Camino medio: 3.4720343758805297
  Diámetro: 8

Componente 2 (3 nodos): ['MULLY', 'KEGS', 'BLACK_JACK']
  Camino medio: 1.0
  Diámetro: 1


<h3>Resultados de la temporada 4</h3>

<p><b>Estructura.</b> La red alcanza su máximo tamaño por temporada con 172 personajes y 667 aristas, lo que reduce su densidad a 0,045. El grafo <b>no es conexo</b> y presenta 2 componentes: una componente principal de 169 personajes (camino medio de 3,47 y diámetro de 8, el más amplio de toda la serie) y una pequeña componente aislada de 3 personajes (Mully, Kegs y Black Jack, miembros de la Guardia de la Noche que solo interactúan entre sí en una escena, con camino medio y diámetro de 1,0).</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Joffrey (40)</td><td>Tyrion (876)</td><td>Joffrey (0,205)</td><td>Tyrion (0,484)</td><td>Joffrey (0,452)</td></tr>
  <tr><td>2</td><td>Tyrion (38)</td><td>Cersei (619)</td><td>Ned (0,194)</td><td>Jaime (0,435)</td><td>Jaime (0,428)</td></tr>
  <tr><td>3</td><td>Cersei (33)</td><td>Jaime (598)</td><td>Stannis (0,178)</td><td>Tywin (0,397)</td><td>Ned (0,420)</td></tr>
  <tr><td>4</td><td>Sansa (31)</td><td>Tywin (586)</td><td>Jon (0,146)</td><td>Cersei (0,386)</td><td>Cersei (0,410)</td></tr>
  <tr><td>5</td><td>Jon (30)</td><td>Jon (542)</td><td>Varys (0,108)</td><td>Oberyn (0,270)</td><td>Tyrion (0,407)</td></tr>
</table>
<p><i>Nota: Sam y Oberyn empatan a 25 contactos en grado, y Sansa y Oberyn empatan a 468 interacciones en grado ponderado.</i></p>

<p><b>Los protagonistas.</b> Desembarco del Rey monopoliza la temporada alrededor de la Boda Púrpura y el juicio a Tyrion. Joffrey encabeza las métricas estructurales (grado de 40, intermediación de 0,205 y cercanía de 0,452) debido a la gran concentración de personajes que asisten a su boda. Sin embargo, en grado ponderado (876) y autovector (0,484) el protagonista indiscutible es Tyrion, seguido por el bloque Lannister-Martell (Cersei, Jaime, Tywin y Oberyn), reflejando la intensidad de los diálogos de su encarcelamiento y juicio.</p>

<p><b>Grado frente a grado ponderado.</b> Joffrey es el ejemplo más claro de sesgo de reunión: primero en grado (40 contactos), pero octavo en grado ponderado (376) al morir al principio de la temporada. En el extremo opuesto, Daenerys es novena en grado ponderado (349) sin entrar en el top 10 de grado, concentrando sus escenas en la conquista de Meereen.</p>

<p><b>Puentes entre grupos.</b> Joffrey (0,205), Ned (0,194) y Stannis (0,178) lideran la intermediación. Stannis experimenta una subida notable al desplazarse desde Rocadragón hasta el Muro para derrotar a los salvajes, conectando la trama política del sur con la de Jon Nieve (cuarto, 0,146). Barristan (sexto, 0,103) vuelve a ejercer de puente entre Essos y Poniente.</p>

<p><b>Comunidades.</b> Se detectan 41 comunidades (10 unipersonales). Destacan:</p>
<ul>
  <li><b>Más allá del Muro (motín de Craster y Cuervo de Tres Ojos):</b> Bran (aislado), Hodor, Jojen, Meera, Three-Eyed Raven, Night King, Leaf, Craster, Karl Tanner, Locke y Jeor (15 personajes).</li>
  <li><b>El Norte y los Bolton:</b> Roose Bolton, Ramsay, Theon, Yara, Rickon, Myranda y Walda (15 personajes).</li>
  <li><b>Boda Púrpura:</b> Joffrey, Loras, High Septon, Baelor y los cinco enanos bufones de la boda.</li>
  <li><b>Viaje de Arya y el Perro:</b> Hound, Polliver, Rorge, Biter, Lommy y campesinos del camino.</li>
  <li><b>El Muro y Rocadragón:</b> Stannis, Davos, Melisandre, Shireen, Selyse, Maester Aemon y Eddison Tollett.</li>
</ul>

<h3>Análisis grafo temporada 5</h3>

<p>Grado</p>

In [72]:
grado = dict(G_ep5.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('CERSEI', 30), ('JON', 25), ('LITTLEFINGER', 25), ('STANNIS', 24), ('SANSA', 21), ('SAM', 20), ('TYRION', 19), ('JAIME', 17), ('DAENERYS', 14), ('BRONN', 12)]


<p>Grado ponderado.</p>

In [73]:
grado_pond = dict(G_ep5.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('CERSEI', 590), ('JON', 552), ('TYRION', 480), ('DAENERYS', 413), ('STANNIS', 390), ('SANSA', 361), ('JAIME', 328), ('SAM', 313), ('LITTLEFINGER', 290), ('JORAH', 244)]


<p>Intermediación (betweenness)</p>

In [74]:
inter = nx.betweenness_centrality(G_ep5)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('STANNIS', 0.3239070538813891), ('CERSEI', 0.26703703820377966), ('LITTLEFINGER', 0.1892065347931807), ('JON', 0.15599253536610638), ('SANSA', 0.12995905219448908), ('TYRION', 0.12635168545615638), ('JAIME', 0.12096204961096668), ('ROOSE_BOLTON', 0.11774408375346056), ('ROBERT', 0.10416926329293576), ('SAM', 0.07408303665365813)]


<p>Autovector (eigenvector)</p>

In [75]:
auto = nx.eigenvector_centrality(G_ep5, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.5910673812477145), ('DAENERYS', 0.4734895216570864), ('JORAH', 0.4331548354437908), ('VARYS', 0.27888519660847516), ('DAARIO', 0.27416955416752153), ('HIZDAHR', 0.1947921622350885), ('MISSANDEI', 0.12382239246134492), ('GREY_WORM', 0.08097147727038763), ('BARRISTAN', 0.08051756224419675), ('MOSSADOR', 0.07290489654830742)]


<p>Cercanía (closeness)</p>

In [76]:
cerc = nx.closeness_centrality(G_ep5)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('CERSEI', 0.4591439688715953), ('LITTLEFINGER', 0.4555984555984556), ('STANNIS', 0.4521072796934866), ('SANSA', 0.44528301886792454), ('ROOSE_BOLTON', 0.4354243542435424), ('ROBERT', 0.4244604316546763), ('PODRICK', 0.4068965517241379), ('JAIME', 0.40273037542662116), ('NED', 0.40273037542662116), ('TYWIN', 0.4)]


<p>Comunidades</p>

In [77]:
comunidades = nx.community.greedy_modularity_communities(G_ep5, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

41
['ARYA', 'MADAME', 'MACE', 'MOUNTAIN', 'THIN_MAN', 'WALDER', 'TYCHO', 'MERYN_TRANT']
['PYP', 'GRENN', 'JANOS', 'LITTLE_SAM', 'ALLISER_THORNE', 'DENYS', 'OLLY', 'EDDISON_TOLLETT']
['MELARA', 'RENLY', 'MAGGY', 'JOFFREY', 'SHAE', 'RHAEGAR', 'TYWIN', 'ROBERT']
['LYSA', 'CATELYN', 'ROYCE', 'ROBIN', 'ROBB', 'NED', 'LITTLEFINGER']
['RANDYLL', 'LORD_WEEBLY', 'SAM', 'MAGNAR', 'BRAND', 'JEOR', 'DERRYK']
['AREO', 'OBERYN', 'TYENE', 'OBARA', 'ELLARIA', 'NYMERIA']
['OLYVAR', 'PYCELLE', 'HIGH_SEPTON', 'KEVAN', 'QYBURN', 'LANCEL']
['LORD_OF_BONES', 'TORMUND', 'KARSI', 'LOBODA', 'NIGHT_KING']
['LOLLYS', 'TANDA', 'BRONN', 'DORNISH_RIDER', 'FALYSE']
['BARRISTAN', 'MOSSADOR', 'GREY_WORM', 'AERYS']
['QUICK', 'STRONG', 'OWNER', 'HIZDAHR']
['RHAENYRA', 'AEGON', 'MAESTER_AEMON', 'GILLY']
['MYRANDA', 'WALDA', 'MAESTER_WOLKAN', 'ROOSE_BOLTON']
['THEON', 'BRAN', 'RICKON']
['BRIAN', 'OTHELL_YARWYCK', 'JON']
['YOHN_ROYCE', 'SANSA', 'LYANNA']
['LYANNA_MORMONT', 'STANNIS', 'STEFFON']
['TRYSTANE', 'DORAN']
['BLAC

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [78]:
print(
    f"nodos: {G_ep5.number_of_nodes()} aristas: {G_ep5.number_of_edges()}"
    f" densidad: {nx.density(G_ep5)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep5)}")

nodos: 119 aristas: 396 densidad: 0.05640222190571144
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [79]:
print(nx.average_shortest_path_length(G_ep5))
print(nx.diameter(G_ep5))

3.217347956131605
8


<h3>Resultados de la temporada 5</h3>

<p><b>Estructura.</b> La red tiene 119 personajes y 396 aristas, con una densidad de 0,056. Es un grafo conexo (1 sola componente), con un camino medio de 3,22 y un diámetro de 8. Pese a la dispersión de las tramas en Braavos, Dorne, Meereen, Invernalia y el Muro, toda la red permanece conectada en una única componente.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Cersei (30)</td><td>Cersei (590)</td><td>Stannis (0,324)</td><td>Tyrion (0,591)</td><td>Cersei (0,459)</td></tr>
  <tr><td>2</td><td>Jon (25)</td><td>Jon (552)</td><td>Cersei (0,267)</td><td>Daenerys (0,473)</td><td>Littlefinger (0,456)</td></tr>
  <tr><td>3</td><td>Littlefinger (25)</td><td>Tyrion (480)</td><td>Littlefinger (0,189)</td><td>Jorah (0,433)</td><td>Stannis (0,452)</td></tr>
  <tr><td>4</td><td>Stannis (24)</td><td>Daenerys (413)</td><td>Jon (0,156)</td><td>Varys (0,279)</td><td>Sansa (0,445)</td></tr>
  <tr><td>5</td><td>Sansa (21)</td><td>Stannis (390)</td><td>Sansa (0,130)</td><td>Daario (0,274)</td><td>Roose Bolton (0,435)</td></tr>
</table>
<p><i>Nota: Jon y Littlefinger empatan a 25 contactos en grado, y Jaime y Ned empatan en cercanía (0,403).</i></p>

<p><b>Protagonismo fragmentado según la métrica.</b> Cersei domina el grado (30), el grado ponderado (590) y la cercanía (0,459) a raíz de su conflicto con la Fe Militante y su paseo de la penitencia en Desembarco del Rey. Jon es segundo en grado (25) y grado ponderado (552) como Lord Comandante de la Guardia de la Noche. Sin embargo, el autovector está completamente copado por el núcleo de Essos: Tyrion (0,591), Daenerys (0,473), Jorah (0,433), Varys (0,279) y Daario (0,274) ocupan los cinco primeros puestos gracias al encuentro entre Tyrion y Daenerys en Meereen.</p>

<p><b>Grado frente a grado ponderado.</b> Tyrion (séptimo en grado con 19 contactos) y Daenerys (novena con 14) escalan al tercer y cuarto puesto en grado ponderado (480 y 413), mientras que Jorah entra en el top 10 ponderado (244) sin figurar en grado: sus viajes juntos generan un volumen muy alto de interacciones concentradas. En sentido opuesto, Littlefinger es tercero en grado (25) pero cae al noveno puesto en grado ponderado (290).</p>

<p><b>Puentes entre grupos.</b> Stannis lidera con claridad la intermediación (0,324), por delante de Cersei (0,267) y Littlefinger (0,189). Stannis actúa como bisagra entre el Muro (Jon y Davos) y el Norte controlado por los Bolton, mientras que Littlefinger enlaza el Valle, Invernalia (al entregar a Sansa a los Bolton) y Desembarco del Rey.</p>

<p><b>Comunidades.</b> El algoritmo detecta 41 comunidades (17 unipersonales, entre ellas Daenerys, Davos, Jaime, Cersei, Varys y Margaery). Entre las agrupaciones destacan:</p>
<ul>
  <li><b>Braavos y Casa de Blanco y Negro:</b> Arya, Meryn Trant, Mace Tyrell, Tycho Nestoris, Thin Man y Madame (Jaqen y Waif quedan aislados).</li>
  <li><b>Guardia de la Noche (motín):</b> Alliser Thorne, Olly, Bowen Marsh (implícito en el núcleo), Pyp, Grenn, Janos y Eddison Tollett.</li>
  <li><b>Dorne:</b> Oberyn, Ellaria, Areo Hotah y las Serpientes de Arena (Obara, Nymeria, Tyene).</li>
  <li><b>Casa Austera (Hardhome):</b> Tormund, Karsi, Loboda, Lord of Bones y Night King.</li>
  <li><b>Desembarco del Rey y el Gorrión Supremo:</b> Qyburn, Kevan, Pycelle, Lancel, High Septon y Olyvar (High Sparrow y Septa Unella forman pareja aparte).</li>
</ul>

<h3>Análisis grafo temporada 6</h3>

<p>Grado</p>

In [80]:
grado = dict(G_ep6.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('SANSA', 40), ('JON', 31), ('CERSEI', 27), ('TYRION', 26), ('JAIME', 24), ('DAVOS', 23), ('NED', 23), ('TORMUND', 21), ('RAMSAY', 19), ('DAENERYS', 18)]


<p>Grado ponderado.</p>

In [81]:
grado_pond = dict(G_ep6.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 598), ('SANSA', 549), ('TYRION', 442), ('JAIME', 407), ('DAVOS', 394), ('CERSEI', 329), ('DAENERYS', 289), ('ARYA', 275), ('BRAN', 273), ('MARGAERY', 264)]


<p>Intermediación (betweenness)</p>

In [82]:
inter = nx.betweenness_centrality(G_ep6)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('SANSA', 0.26262369597304924), ('JON', 0.19134542283724268), ('TYRION', 0.14853156113828953), ('JAIME', 0.13044955131298), ('CERSEI', 0.11721858119776939), ('OLENNA', 0.11357549660741154), ('NED', 0.11267022269017848), ('SAM', 0.10229989868287741), ('THEON', 0.0857122742476336), ('DAENERYS', 0.08319682522132986)]


<p>Autovector (eigenvector)</p>

In [83]:
auto = nx.eigenvector_centrality(G_ep6, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 0.5563531729923052), ('SANSA', 0.4857279777446351), ('DAVOS', 0.3946420666638599), ('MELISANDRE', 0.26447140957988413), ('TORMUND', 0.2633574552481489), ('EDDISON_TOLLETT', 0.19174686212543948), ('RAMSAY', 0.16088328199668822), ('BRIENNE', 0.15989868151921702), ('LITTLEFINGER', 0.12962039507862533), ('LYANNA_MORMONT', 0.09695691829566283)]


<p>Cercanía (closeness)</p>

In [84]:
cerc = nx.closeness_centrality(G_ep6)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('SANSA', 0.48437799501629286), ('TYRION', 0.4296366462644516), ('JON', 0.4238307456392563), ('CERSEI', 0.4181796690307329), ('JAIME', 0.4140392762680523), ('NED', 0.40599967867061443), ('ARYA', 0.404690002287806), ('MARGAERY', 0.3995347156344582), ('BRIENNE', 0.395753629997539), ('PODRICK', 0.3920434397163121)]


<p>Comunidades</p>

In [85]:
comunidades = nx.community.greedy_modularity_communities(G_ep6, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

37
['HOSTER', 'BRYNDEN', 'MINISA', 'BRONN', 'EDMURE', 'SON_OF_EDMURE', 'WALDER', 'LOTHAR', 'BLACK_WALDER', 'GUARD_CAPTAIN']
['ARTHUR', 'RODRIK', 'LYANNA', 'THREE_EYED_RAVEN', 'HODOR', 'OLD_NAN', 'HOWLAND', 'NED', 'RICKARD_STARK', 'BENJEN']
['TRYSTANE', 'AREO', 'OBERYN', 'TYENE', 'OLENNA', 'ELLARIA', 'MAESTER_DORNE', 'OBARA', 'NYMERIA', 'DORAN']
['BOBONO', 'IZEMBARO', 'CAMELLO', 'LADY_CRANE', 'CLARENZO', 'JOFFREY', 'BIANCA', 'TYWIN', 'ROBERT']
['RANDYLL', 'MAESTER_CITADEL', 'MAESTER_AEMON', 'SAM', 'LITTLE_SAM', 'TALLA', 'GILLY', 'DICKON', 'MELESSA']
['LEM', 'BERIC', 'FLYNN', 'RAY', 'MORGAN', 'HOUND', 'GATINS', 'THOROS']
['LORAS', 'MACE', 'ARTHUR_CHILD', 'PYCELLE', 'KEVAN', 'QYBURN', 'LANCEL']
['CATELYN', 'MAESTER_MORMONT', 'ROBB', 'GLOVER', 'LYANNA_MORMONT', 'MANDERLY', 'ROBETT']
['WALDA', 'SON_OFWALDA', 'ROOSE_BOLTON', 'MAESTER_WOLKAN', 'RAMSAY', 'MYRANDA', 'RICKARD_KARSTARK']
['JORAH', 'DAARIO', 'DOSH_KHALEEN_LEADER', 'DROGO', 'KHAL_MORO', 'LHAZAREEN_WOMAN']
['RICKON', 'OSHA', 'WUN_WU

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [86]:
print(
    f"nodos: {G_ep6.number_of_nodes()} aristas: {G_ep6.number_of_edges()}"
    f" densidad: {nx.density(G_ep6)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep6)}")

nodos: 142 aristas: 541 densidad: 0.05404055538907202
componentes conexas: 2


<p>Camino medio y diámetro.</p>

In [126]:
calcular_caminos(G_ep6)

El grafo NO es conexo (2 componentes).

Componente 1 - Principal (134 nodos):
  Camino medio: 3.171024576366289
  Diámetro: 7

Componente 2 (8 nodos): ['LEM', 'BERIC', 'FLYNN', 'RAY', 'MORGAN', 'HOUND', 'GATINS', 'THOROS']
  Camino medio: 1.5714285714285714
  Diámetro: 3


<h3>Resultados de la temporada 6</h3>

<p><b>Estructura.</b> La red tiene 142 personajes y 541 aristas, con una densidad de 0,054. El grafo <b>no es conexo</b> y se divide en 2 componentes: una componente principal de 134 personajes (camino medio de 3,17 y diámetro de 7) y una componente aislada de 8 personajes correspondiente a la trama del Perro con la Hermandad sin Estandartes (Hound, Beric, Thoros, Lem, Flynn, Ray, Morgan y Gatins, con camino medio de 1,57 y diámetro de 3), cuya historia transcurre al margen del resto de tramas durante esta temporada.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Sansa (40)</td><td>Jon (598)</td><td>Sansa (0,263)</td><td>Jon (0,556)</td><td>Sansa (0,484)</td></tr>
  <tr><td>2</td><td>Jon (31)</td><td>Sansa (549)</td><td>Jon (0,191)</td><td>Sansa (0,486)</td><td>Tyrion (0,430)</td></tr>
  <tr><td>3</td><td>Cersei (27)</td><td>Tyrion (442)</td><td>Tyrion (0,149)</td><td>Davos (0,395)</td><td>Jon (0,424)</td></tr>
  <tr><td>4</td><td>Tyrion (26)</td><td>Jaime (407)</td><td>Jaime (0,130)</td><td>Melisandre (0,264)</td><td>Cersei (0,418)</td></tr>
  <tr><td>5</td><td>Jaime (24)</td><td>Davos (394)</td><td>Cersei (0,117)</td><td>Tormund (0,263)</td><td>Jaime (0,414)</td></tr>
</table>
<p><i>Nota: Davos y Ned empatan a 23 contactos en grado.</i></p>

<p><b>El protagonismo en el Norte.</b> Sansa y Jon se reparten el liderazgo absoluto de la temporada en torno a la recuperación de Invernalia y la Batalla de los Bastardos. Sansa encabeza las métricas estructurales: grado (40 contactos), intermediación (0,263) y cercanía (0,484), al reunir apoyos entre las casas norteñas, los salvajes y el Valle. Jon lidera el grado ponderado (598) y el autovector (0,556), seguido muy de cerca por Sansa (549 y 0,486) y Davos (394 y 0,395).</p>

<p><b>Grado frente a grado ponderado.</b> Arya (275), Bran (273) y Margaery (264) aparecen en el top 10 de grado ponderado sin entrar entre los diez primeros por grado: sus tramas (Braavos, las visiones del Cuervo de Tres Ojos y el cautiverio en el Septo de Baelor) concentran muchas escenas con un grupo reducido de personajes. Por el contrario, Tormund (21 contactos) y Ramsay (19) destacan en grado por escenas grupales de guerra, pero quedan fuera del top 10 ponderado.</p>

<p><b>Puentes entre grupos.</b> Sansa (0,263) y Jon (0,191) son los principales conectores de la red, seguidos de Tyrion (0,149) y Jaime (0,130). Destaca Olenna Tyrell en sexto puesto (0,114), actuando como enlace entre Desembarco del Rey y Dorne al pactar con Ellaria Arena contra Cersei.</p>

<p><b>Comunidades.</b> Se detectan 37 comunidades (11 unipersonales, entre ellas Jon —implícito en el núcleo norteño—, Sansa, Arya, Bran, Daenerys, Jaime y Cersei). Entre los grupos principales destacan:</p>
<ul>
  <li><b>Visiones de Bran y pasado Stark:</b> Ned, Lyanna, Benjen, Howland Reed, Arthur Dayne, Hodor, Old Nan, Rodrik y Three-Eyed Raven.</li>
  <li><b>Asedio de Aguasdulces y los Gemelos:</b> Jaime (aislado), Bronn, Brienne (aislada), Brynden Tully, Edmure, Walder Frey, Lothar y Black Walder.</li>
  <li><b>Braavos (compañía teatral):</b> Lady Crane, Izembaro, Bobono, Camello, Clarenzo y Bianca (Arya y Waif quedan fuera de este bloque).</li>
  <li><b>Colina Cuerno y la Ciudadela:</b> Sam, Gilly, Little Sam, Randyll, Dickon, Melessa y Talla Tarly.</li>
  <li><b>Islas del Hierro:</b> Euron, Yara, Aeron y Balon Greyjoy.</li>
</ul>

<h3>Análisis grafo temporada 7</h3>

<p>Grado</p>

In [88]:
grado = dict(G_ep7.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 43), ('DAENERYS', 34), ('TYRION', 34), ('CERSEI', 31), ('DAVOS', 27), ('SANSA', 24), ('BRIENNE', 24), ('THEON', 24), ('HOUND', 23), ('JORAH', 22)]


<p>Grado ponderado.</p>

In [89]:
grado_pond = dict(G_ep7.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 968), ('DAENERYS', 733), ('TYRION', 693), ('CERSEI', 588), ('SANSA', 542), ('JAIME', 521), ('DAVOS', 361), ('HOUND', 327), ('ARYA', 317), ('JORAH', 307)]


<p>Intermediación (betweenness)</p>

In [90]:
inter = nx.betweenness_centrality(G_ep7)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 0.237362397781175), ('DAENERYS', 0.11515170195132578), ('CERSEI', 0.11205674388253957), ('SAM', 0.10441836533706125), ('SANSA', 0.09108866886217318), ('TYRION', 0.08310372822923297), ('BRAN', 0.06815847971192848), ('JORAH', 0.0495849896207265), ('THEON', 0.04835906640452011), ('ARYA', 0.03888085859226906)]


<p>Autovector (eigenvector)</p>

In [91]:
auto = nx.eigenvector_centrality(G_ep7, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 0.48509868017485575), ('DAENERYS', 0.48228867265056213), ('TYRION', 0.4435330327832724), ('CERSEI', 0.24437001017666904), ('DAVOS', 0.22777117316533577), ('JAIME', 0.19602165651059147), ('SANSA', 0.15893056801186528), ('VARYS', 0.15752973384042918), ('JORAH', 0.15350778578218588), ('MISSANDEI', 0.14132947874538127)]


<p>Cercanía (closeness)</p>

In [92]:
cerc = nx.closeness_centrality(G_ep7)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('JON', 0.6611570247933884), ('TYRION', 0.6015037593984962), ('DAENERYS', 0.5882352941176471), ('CERSEI', 0.5755395683453237), ('DAVOS', 0.5594405594405595), ('BRIENNE', 0.547945205479452), ('SANSA', 0.5369127516778524), ('JAIME', 0.5298013245033113), ('THEON', 0.5298013245033113), ('HOUND', 0.5228758169934641)]


<p>Comunidades</p>

In [93]:
comunidades = nx.community.greedy_modularity_communities(G_ep7, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

34
['BRAN', 'LYANNA', 'RHAEGAR', 'HOWLAND', 'MEERA', 'EDDISON_TOLLETT']
['HIGH_SEPTON', 'SAM', 'LITTLE_SAM', 'STANNIS', 'GILLY', 'JEOR']
['YOHN_ROYCE', 'PODRICK', 'LYANNA_MORMONT', 'QHONO', 'ROBETT']
['YARA', 'TYENE', 'ELLARIA', 'OBARA', 'NYMERIA']
['TOMMEN', 'ROBB', 'JOFFREY', 'NED', 'ROBERT']
['NED_UMBER', 'JON', 'ALYS', 'BENJEN']
['LYSA', 'JON_ARRYN', 'RAMSAY', 'SANSA']
['MOUNTAIN', 'MYRCELLA', 'OBERYN']
['DAENERYS', 'AEGON', 'VISERYS']
['TYWIN', 'JOANNA', 'TYCHO']
['OLENNA', 'MARGAERY', 'AERYS']
['SHIREEN', 'MARWYN', 'MAESTER_WOLKAN']
['HOT_PIE', 'BRIENNE']
['LITTLEFINGER', 'CATELYN']
['ARYA', 'WALDER']
['QYBURN', 'BALERION']
['DICKON', 'RANDYLL']
['DROGO', 'JORAH']
['MELISANDRE', 'GREY_WORM']
['HARRAG', 'THEON']
['TORMUND', 'WHITE_WALKER']
['CERSEI']
['TYRION']
['VARYS']
['BERIC']
['DAVOS']
['GENDRY']
['HOUND']
['NIGHT_KING']
['THOROS']
['BRONN']
['EURON']
['JAIME']
['MISSANDEI']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [94]:
print(
    f"nodos: {G_ep7.number_of_nodes()} aristas: {G_ep7.number_of_edges()}"
    f" densidad: {nx.density(G_ep7)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep7)}")

nodos: 81 aristas: 412 densidad: 0.1271604938271605
componentes conexas: 1


<p>Camino medio y diámetro.</p>

In [95]:
print(nx.average_shortest_path_length(G_ep7))
print(nx.diameter(G_ep7))

2.3722222222222222
5


<h3>Resultados de la temporada 7</h3>

<p><b>Estructura.</b> La red se reduce drásticamente a 81 personajes y 412 aristas, duplicando su densidad hasta 0,127. Es un grafo conexo (1 sola componente), con un camino medio de 2,37 y un diámetro de 5. La llegada de Daenerys a Poniente y la cumbre en Pozo Dragón provocan la convergencia de casi todas las tramas principales en una red mucho más compacta.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Jon (43)</td><td>Jon (968)</td><td>Jon (0,237)</td><td>Jon (0,485)</td><td>Jon (0,661)</td></tr>
  <tr><td>2</td><td>Daenerys (34)</td><td>Daenerys (733)</td><td>Daenerys (0,115)</td><td>Daenerys (0,482)</td><td>Tyrion (0,602)</td></tr>
  <tr><td>3</td><td>Tyrion (34)</td><td>Tyrion (693)</td><td>Cersei (0,112)</td><td>Tyrion (0,444)</td><td>Daenerys (0,588)</td></tr>
  <tr><td>4</td><td>Cersei (31)</td><td>Cersei (588)</td><td>Sam (0,104)</td><td>Cersei (0,244)</td><td>Cersei (0,576)</td></tr>
  <tr><td>5</td><td>Davos (27)</td><td>Sansa (542)</td><td>Sansa (0,091)</td><td>Davos (0,228)</td><td>Davos (0,559)</td></tr>
</table>
<p><i>Nota: Daenerys y Tyrion empatan a 34 contactos en grado; Sansa, Brienne y Theon empatan a 24 contactos; y Jaime y Theon empatan en cercanía (0,530).</i></p>

<p><b>El protagonista.</b> Jon Nieve lidera las cinco métricas por primera vez desde Ned Stark en la primera temporada. Comparte escena con 43 de los otros 80 personajes (un 53,8 %) y acumula 968 interacciones ponderadas. Su viaje desde Invernalia a Rocadragón, la expedición más allá del Muro y la reunión en Desembarco del Rey lo sitúan en el centro absoluto de la red, formando un trío dominante junto a Daenerys y Tyrion.</p>

<p><b>Grado frente a grado ponderado.</b> Jaime (521) y Arya (317) entran en el top 10 de grado ponderado sin estar entre los diez primeros por grado: Arya concentra sus escenas en Invernalia con Sansa y Littlefinger, mientras que Jaime lo hace en la capital y en la batalla del Camino de Rosas. En sentido opuesto, Brienne (24 contactos) y Theon (24) destacan en grado por su presencia en reuniones diplomáticas (como Pozo Dragón), pero quedan fuera del top 10 ponderado.</p>

<p><b>Puentes entre grupos.</b> Jon (0,237) duplica en intermediación a Daenerys (0,115) y Cersei (0,112). Destaca el cuarto puesto de Sam (0,104), quien conecta la trama de la Ciudadela (donde cura a Jorah y descubre la legitimidad de Jon) con Invernalia y el resto de la red.</p>

<p><b>Comunidades.</b> Se detectan 34 comunidades (13 unipersonales, entre ellas Cersei, Tyrion, Varys, Davos, Jaime y Hound). Entre las agrupaciones destacan:</p>
<ul>
  <li><b>Visiones de Bran y el Norte:</b> Bran, Meera, Eddison Tollett, Lyanna, Rhaegar y Howland Reed.</li>
  <li><b>La Ciudadela:</b> Sam, Gilly, Little Sam, High Septon, Stannis y Jeor.</li>
  <li><b>Señores del Norte y el Valle:</b> Yohn Royce, Lyanna Mormont, Robett Glover, Podrick y Qhono.</li>
  <li><b>Flota Greyjoy y Dorne:</b> Yara, Ellaria, Obara, Nymeria y Tyene.</li>
</ul>

<h3>Análisis grafo temporada 8</h3>

<p>Grado</p>

In [134]:
grado = dict(G_ep8.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('SAM', 42), ('DAENERYS', 41), ('TYRION', 41), ('SANSA', 40), ('ARYA', 39), ('GREY_WORM', 38), ('DAVOS', 37), ('JON', 36), ('BRIENNE', 34), ('JAIME', 32)]


<p>Grado ponderado.</p>

In [135]:
grado_pond = dict(G_ep8.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 1062), ('JON', 914), ('DAENERYS', 843), ('JAIME', 586), ('SANSA', 577), ('ARYA', 509), ('BRIENNE', 476), ('SAM', 461), ('GREY_WORM', 454), ('TORMUND', 417)]


<p>Intermediación (betweenness)</p>

In [136]:
inter = nx.betweenness_centrality(G_ep8)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('DAENERYS', 0.13390159981644), ('SAM', 0.0991329119607736), ('ARYA', 0.09902087596468989), ('BRONN', 0.0824013069403841), ('TYRION', 0.06311681962029121), ('SANSA', 0.05999541827707136), ('JON', 0.056995119101493824), ('JAIME', 0.052323285996643445), ('VARYS', 0.0476474802868322), ('BRIENNE', 0.04046177950249275)]


<p>Autovector (eigenvector)</p>

In [137]:
auto = nx.eigenvector_centrality(G_ep8, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.4493407205070147), ('JON', 0.4305205328607398), ('DAENERYS', 0.40821837620012696), ('SANSA', 0.2684766432061161), ('JAIME', 0.24074444006599127), ('VARYS', 0.2181818881527782), ('GREY_WORM', 0.18625745328430937), ('BRIENNE', 0.18275972331108734), ('DAVOS', 0.1775608242298798), ('SAM', 0.17092175924076974)]


<p>Cercanía (closeness)</p>

In [138]:
cerc = nx.closeness_centrality(G_ep8)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('SAM', 0.6770077894171367), ('DAENERYS', 0.670434898257747), ('TYRION', 0.670434898257747), ('DAVOS', 0.6453719114069901), ('ARYA', 0.6393962455606291), ('SANSA', 0.6393962455606291), ('BRIENNE', 0.6277708592777086), ('GREY_WORM', 0.6277708592777086), ('JON', 0.6165606653620352), ('JAIME', 0.6165606653620352)]


<p>Comunidades</p>

In [139]:
comunidades = nx.community.greedy_modularity_communities(G_ep8, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

31
['YOHN_ROYCE', 'YARA', 'ROBIN', 'EDMURE', 'VALE_LORD', 'MAESTER_WOLKAN', 'RIVERLANDS_LORD', 'DORNISH_PRINCE', 'IRONBORN_LORD']
['WILLIAM', 'BRONN', 'CRAYAH', 'JOFFREY', 'DIRAH', 'MAREI']
['BERIC', 'NED_UMBER', 'TEELA', 'GILLY', 'EDDISON_TOLLETT']
['VISERYS', 'DAENERYS', 'AEGON', 'AERYS']
['LYANNA', 'RHAEGAR', 'NED', 'ROBERT']
['MARWYN', 'DICKON', 'SAM', 'RANDYLL']
['ARYA', 'OWEN', 'ALANNA']
['TYWIN', 'CERSEIS_BABY', 'CERSEI']
['HOUND', 'WILLA', 'SARRA']
['ALTON', 'JAIME']
['QHONO', 'MELISANDRE']
['NIGHT_KING', 'THEON']
['VICKY', 'NORA']
['CATELYN', 'BRIENNE']
['MOUNTAIN', 'QYBURN']
['EURON', 'HARRY']
['LITTLE_SAM', 'PODRICK']
['UNSULLIED_CAPTAIN', 'GREY_WORM']
['LITTLEFINGER', 'RAMSAY']
['MARTHA', 'VARYS']
['JON']
['ALYS']
['BRAN']
['DAVOS']
['GENDRY']
['JORAH']
['LYANNA_MORMONT']
['MISSANDEI']
['SANSA']
['TORMUND']
['TYRION']


<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [140]:
print(
    f"nodos: {G_ep8.number_of_nodes()} aristas: {G_ep8.number_of_edges()}"
    f" densidad: {nx.density(G_ep8)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_ep8)}")

nodos: 74 aristas: 553 densidad: 0.20473898556090336
componentes conexas: 2


<p>Camino medio y diámetro.</p>

In [142]:
calcular_caminos(G_ep8)

El grafo NO es conexo (2 componentes).

Componente 1 - Principal (72 nodos):
  Camino medio: 2.064162754303599
  Diámetro: 6

Componente 2 (2 nodos): ['LITTLEFINGER', 'RAMSAY']
  Camino medio: 1.0
  Diámetro: 1


<h3>Resultados de la temporada 8</h3>

<p><b>Estructura.</b> Es la red más pequeña y más densa de toda la serie, con 74 personajes y 553 aristas, alcanzando una densidad de 0,205 (más del triple que en las primeras temporadas). El grafo <b>no es conexo</b> y presenta 2 componentes: una componente principal de 72 personajes (camino medio de 2,06 y diámetro de 6) y una pareja aislada formada por Littlefinger y Ramsay (camino medio y diámetro de 1,0), correspondiente a una mención o flashback desconectado del presente narrativo de la temporada.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Sam (42)</td><td>Tyrion (1062)</td><td>Daenerys (0,134)</td><td>Tyrion (0,449)</td><td>Sam (0,677)</td></tr>
  <tr><td>2</td><td>Daenerys (41)</td><td>Jon (914)</td><td>Sam (0,099)</td><td>Jon (0,431)</td><td>Daenerys (0,670)</td></tr>
  <tr><td>3</td><td>Tyrion (41)</td><td>Daenerys (843)</td><td>Arya (0,099)</td><td>Daenerys (0,408)</td><td>Tyrion (0,670)</td></tr>
  <tr><td>4</td><td>Sansa (40)</td><td>Jaime (586)</td><td>Bronn (0,082)</td><td>Sansa (0,268)</td><td>Davos (0,645)</td></tr>
  <tr><td>5</td><td>Arya (39)</td><td>Sansa (577)</td><td>Tyrion (0,063)</td><td>Jaime (0,241)</td><td>Arya (0,639)</td></tr>
</table>
<p><i>Nota: Daenerys y Tyrion empatan en grado (41) y en cercanía (0,670); Arya y Sansa empatan en cercanía (0,639); Brienne y Grey Worm empatan en cercanía (0,628); y Jon y Jaime empatan en cercanía (0,617).</i></p>

<p><b>El protagonismo depende de la métrica.</b> La convergencia de casi todos los supervivientes en Invernalia (para la Larga Noche) y posteriormente en Desembarco del Rey genera un reparto entre dos perfiles. Tyrion lidera el grado ponderado (1062) y el autovector (0,449), seguido de Jon (914 y 0,431) y Daenerys (843 y 0,408), concentrando el mayor volumen de diálogo dramático. Sin embargo, en grado (42) y cercanía (0,677) encabeza la tabla Sam, gracias a su presencia transversal en las reuniones estratégicas de Invernalia y en el consejo final de Pozo Dragón.</p>

<p><b>Grado frente a grado ponderado.</b> Sam (primero en grado con 42 contactos) cae al octavo puesto en grado ponderado (461), y Davos (séptimo en grado con 37) ni siquiera entra en el top 10 ponderado: ambos son ejemplos claros del sesgo de reunión en escenas corales y consejos de guerra. En sentido contrario, Tormund entra en el top 10 ponderado (417) sin figurar en grado, concentrando sus escenas con Jon y el núcleo norteño.</p>

<p><b>Puentes entre grupos.</b> Daenerys lidera la intermediación (0,134), seguida por Sam (0,099), Arya (0,099) y Bronn (0,082). Bronn destaca en cuarta posición al actuar como enlace directo entre Desembarco del Rey (encargado por Cersei/Qyburn) e Invernalia (donde confronta a Jaime y Tyrion).</p>

<p><b>Comunidades.</b> Se detectan 31 comunidades (11 unipersonales, como Jon, Tyrion, Sansa, Bran, Davos y Jorah). Entre las agrupaciones principales destacan:</p>
<ul>
  <li><b>Consejo final de Poniente:</b> Yohn Royce, Robin Arryn, Edmure Tully, Yara Greyjoy, Maester Wolkan y los señores del Valle, Tierras de los Ríos, Dorne e Islas del Hierro (9 personajes).</li>
  <li><b>Desembarco del Rey (taberna y entorno de Bronn):</b> Bronn, Marei, Cayah, Dirah, William y Joffrey.</li>
  <li><b>El Muro y Último Hogar:</b> Beric, Eddison Tollett, Gilly, Ned Umber y Teela.</li>
  <li><b>Dinastía Targaryen:</b> Daenerys, Viserys, Aegon y Aerys.</li>
</ul>

<h3>Análisis grafo general (todas las temporadas)</h3>

<p>Grado</p>

In [96]:
grado = dict(G_general.degree())
print(sorted(grado.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 128), ('JON', 105), ('SANSA', 101), ('ARYA', 97), ('DAENERYS', 93), ('JAIME', 90), ('CERSEI', 86), ('DAVOS', 78), ('NED', 78), ('SAM', 78)]


<p>Grado ponderado.</p>

In [97]:
grado_pond = dict(G_general.degree(weight="weight"))
print(sorted(grado_pond.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 5875), ('JON', 4753), ('DAENERYS', 3839), ('CERSEI', 3727), ('SANSA', 3607), ('JAIME', 3316), ('ARYA', 2791), ('SAM', 2062), ('DAVOS', 1987), ('BRAN', 1864)]


<p>Intermediación (betweenness)</p>

In [98]:
inter = nx.betweenness_centrality(G_general)
print(sorted(inter.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.1319189105755136), ('DAENERYS', 0.11223191310467565), ('ARYA', 0.10183222285305632), ('JON', 0.09555538549242751), ('THEON', 0.06476174934550885), ('SANSA', 0.05686817668253867), ('SAM', 0.056765968319002574), ('JORAH', 0.0558275569432155), ('JAIME', 0.05462480664135093), ('HOUND', 0.048589319319020045)]


<p>Autovector (eigenvector)</p>

In [99]:
auto = nx.eigenvector_centrality(G_general, weight="weight")
print(sorted(auto.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.4830257437323075), ('CERSEI', 0.303990298562118), ('DAENERYS', 0.30283386215920344), ('JON', 0.2868657237532236), ('JAIME', 0.27603868955619176), ('SANSA', 0.26072838853562763), ('VARYS', 0.20875998182910832), ('JORAH', 0.16909790971957497), ('TYWIN', 0.16312807964181708), ('BRONN', 0.15285578379895973)]


<p>Cercanía (closeness)</p>

In [100]:
cerc = nx.closeness_centrality(G_general)
print(sorted(cerc.items(), key=lambda x: x[1], reverse=True)[:10])

[('TYRION', 0.5749599743836055), ('JON', 0.5496173859810224), ('SANSA', 0.5488613785725203), ('JAIME', 0.5392192192192192), ('ARYA', 0.5377657981431566), ('DAENERYS', 0.5320296296296296), ('DAVOS', 0.5292071912761568), ('CERSEI', 0.5285062545989698), ('THEON', 0.5236512102653834), ('BRAN', 0.5215976761074801)]


<p>Comunidades</p>

In [101]:
comunidades = nx.community.greedy_modularity_communities(G_general, weight="weight")
print(len(comunidades))
for c in comunidades:
    print(list(c))

69
['SPICE_KING', 'MALAKKO', 'VISERYS', 'ILLYRIO', 'DROGO', 'IROGENIA', 'MIRRI_MAZ_DUUR', 'RHAEGO', 'SILK_KING', 'PYATT_PREE', 'QOTHO', 'DOREAH', 'COHOLLO', 'WINE_MERCHANT', 'KOVARRO', 'QUAITHE', 'IRRI', 'XARO', 'RAKHARO', 'MAGO', 'JHIQUI']
['MINISA', 'EDMURE', 'RICKARD_KARSTARK', 'JOYEUSE', 'BRYNDEN', 'WILLEM_LANNISTER', 'SON_OF_EDMURE', 'ROSLIN', 'WALDER', 'TORRHEN', 'ALTON', 'LOTHAR', 'BLACK_WALDER', 'FREY_SEPTON', 'MERRY', 'HOSTER', 'TALISA', 'MARTYN', 'RYGER_RIVERS', 'GUARD_CAPTAIN']
['HARRAG', 'ANYA_WAYNWOOD', 'YOHN_ROYCE', 'ROYCE', 'WILL', 'BOLTON_GUARD', 'YARA', 'ROBIN', 'VANCE_CORBRAY', 'MAESTER_WOLKAN', 'VALE_LORD', 'RIVERLANDS_LORD', 'DORNISH_PRINCE', 'IRONBORN_LORD', 'GARED', 'FIRST_MATE', 'WAYMAR_ROYCE']
['PYP', 'HOBB', 'BOWEN_MARSH', 'BORCAS', 'MAESTER_AEMON', 'LUKE', 'ALLISER_THORNE', 'DENYS', 'OLLY', 'JEOR', 'OTHOR', 'DAREON', 'RAST', 'OTHELL_YARWYCK', 'JAREMY_RYKKER']
['ARTHUR', 'CRASTERS_BABY', 'FARMER', 'THREE_EYED_RAVEN', 'HODOR', 'JOJEN', 'ORELL', 'OLD_NAN', 'HOWLA

<p>Estructura: nodos, aristas, densidad y componentes conexas.</p>

In [102]:
print(
    f"nodos: {G_general.number_of_nodes()} aristas: {G_general.number_of_edges()}"
    f" densidad: {nx.density(G_general)}"
)
print(f"componentes conexas: {nx.number_connected_components(G_general)}")

nodos: 406 aristas: 2637 densidad: 0.032074438970990694
componentes conexas: 2


<p>Camino medio y diámetro.</p>

In [127]:
calcular_caminos(G_general)

El grafo NO es conexo (2 componentes).

Componente 1 - Principal (403 nodos):
  Camino medio: 2.6830240855276966
  Diámetro: 6

Componente 2 (3 nodos): ['MULLY', 'KEGS', 'BLACK_JACK']
  Camino medio: 1.0
  Diámetro: 1


<h3>Resultados del grafo general (todas las temporadas)</h3>

<p><b>Estructura.</b> La red global reúne 406 personajes y 2637 aristas, con una densidad de 0,032. El grafo <b>no es conexo</b> y presenta 2 componentes: una componente principal de 403 personajes (camino medio de 2,68 y diámetro de 6) y la misma componente aislada de 3 personajes vista en la cuarta temporada (Mully, Kegs y Black Jack, con camino medio y diámetro de 1,0), que nunca vuelven a interactuar con ningún otro personaje de la serie. Pese a abarcar ocho temporadas y más de cuatrocientos personajes, el camino medio de 2,68 confirma el fenómeno de «mundo pequeño»: cualquier par de personajes de la componente principal está conectado en menos de tres pasos en promedio.</p>

<p><b>Ranking de centralidades (top 5).</b></p>
<table border="1" cellpadding="6" style="border-collapse:collapse">
  <tr><th>Puesto</th><th>Grado</th><th>Grado ponderado</th><th>Intermediación</th><th>Autovector</th><th>Cercanía</th></tr>
  <tr><td>1</td><td>Tyrion (128)</td><td>Tyrion (5875)</td><td>Tyrion (0,132)</td><td>Tyrion (0,483)</td><td>Tyrion (0,575)</td></tr>
  <tr><td>2</td><td>Jon (105)</td><td>Jon (4753)</td><td>Daenerys (0,112)</td><td>Cersei (0,304)</td><td>Jon (0,550)</td></tr>
  <tr><td>3</td><td>Sansa (101)</td><td>Daenerys (3839)</td><td>Arya (0,102)</td><td>Daenerys (0,303)</td><td>Sansa (0,549)</td></tr>
  <tr><td>4</td><td>Arya (97)</td><td>Cersei (3727)</td><td>Jon (0,096)</td><td>Jon (0,287)</td><td>Jaime (0,539)</td></tr>
  <tr><td>5</td><td>Daenerys (93)</td><td>Sansa (3607)</td><td>Theon (0,065)</td><td>Jaime (0,276)</td><td>Arya (0,538)</td></tr>
</table>
<p><i>Nota: Davos, Ned y Sam empatan a 78 contactos en grado (puestos 8 a 10).</i></p>

<p><b>El protagonista de la serie.</b> Tyrion Lannister encabeza con rotundidad las cinco métricas globales: interactúa con 128 personajes distintos (el 31,5 % de todo el reparto) y acumula un grado ponderado de 5875, superando en más de 1100 puntos al segundo clasificado, Jon Nieve (4753). Su liderazgo estructural y ponderado se explica porque es el único personaje principal que transita de forma activa por todos los grandes escenarios de la serie: Invernalia, el Muro, el Nido de Águilas, Desembarco del Rey, Pentos, Meereen y Rocadragón.</p>

<p><b>Grado frente a grado ponderado.</b> En el top 10 global ambos rankings son muy consistentes, pero destacan dos matices. Ned Stark es octavo en grado (78 contactos) pese a morir en la primera temporada, impulsado por su altísima densidad de contactos iniciales y menciones posteriores, pero no entra en el top 10 ponderado. En sentido contrario, Bran es décimo en grado ponderado (1864) sin entrar en el top 10 por grado: mantiene un círculo de acompañantes más reducido, pero con interacciones muy sostenidas a lo largo de toda la serie.</p>

<p><b>Autovector.</b> Tyrion (0,483) lidera con gran ventaja sobre Cersei (0,304), Daenerys (0,303), Jon (0,287), Jaime (0,276) y Sansa (0,261). Destaca la presencia de Varys (séptimo, 0,209), Tywin (noveno, 0,163) y Bronn (décimo, 0,153), quienes no figuran en el top 10 de grado ni de grado ponderado pero alcanzan un alto autovector al concentrar sus interacciones con los nodos más poderosos de Desembarco del Rey y Essos.</p>

<p><b>Los personajes que unen las regiones.</b> En intermediación, tras Tyrion (0,132), destacan Daenerys (0,112), Arya (0,102), Jon (0,096) y Theon (0,065). Theon ocupa el quinto puesto global sin entrar en el top 10 de grado, grado ponderado ni autovector: su recorrido entre Invernalia, las Islas del Hierro, el cautiverio de los Bolton y la flota de Daenerys lo convierte en un puente estructural clave entre subredes periféricas. Lo mismo ocurre con Jorah (octavo, 0,056) y el Perro (décimo, 0,049).</p>

<p><b>Comunidades.</b> El algoritmo detecta 69 comunidades en el grafo global (60 con varios personajes y 9 de un solo elemento: Cersei, Bran, Brienne, Littlefinger, Margaery, Black Haired Prostitute, Loras, Shae y Tommen). Las principales comunidades reflejan fielmente los grandes bloques geográficos y narrativos de la serie:</p>
<ul>
  <li><b>Essos y Qarth (primeras temporadas):</b> Drogo, Viserys, Illyrio, Xaro, Pyat Pree, Quaithe, Mirri Maz Duur, Doreah, Irri, Rakharo y Jorah (21 personajes; Daenerys queda en un subgrupo propio de Meereen con Fennesz y Master Mighdal).</li>
  <li><b>Tierras de los Ríos, Tully y Frey:</b> Walder Frey, Edmure, Brynden, Hoster, Talisa, Rickard Karstark, Lothar y Black Walder (20 personajes).</li>
  <li><b>Señores de Poniente y Consejo Final:</b> Yohn Royce, Robin Arryn, Yara Greyjoy, Maester Wolkan, Waymar Royce, Gared, Will y los señores regionales (17 personajes).</li>
  <li><b>Guardia de la Noche:</b> Jeor, Maester Aemon, Alliser Thorne, Olly, Pyp, Bowen Marsh, Othell Yarwyck y Rast (15 personajes).</li>
  <li><b>Más allá del Muro y Caminantes Blancos:</b> Night King, White Walker, Three-Eyed Raven, Hodor, Jojen, Meera, Leaf, Old Nan y Howland Reed (13 personajes).</li>
  <li><b>Meereen y los Hijos de la Arpía:</b> Daario, Hizdahr, Mero, Prendahl y Khal Moro (13 personajes).</li>
  <li><b>Viajes por las Tierras de los Ríos (el Perro):</b> Hound, Polliver, Lommy, Gatins, Ray y posaderos del camino (13 personajes).</li>
  <li><b>Ejército del Norte (Rey en el Norte):</b> Robb, Greatjon Umber, Manderly, Glover, Jonos Bracken y Galbart Glover (11 personajes).</li>
  <li><b>Dorne:</b> Doran, Oberyn, Ellaria, Areo Hotah, Myrcella, Trystane y las Serpientes de Arena (9 personajes).</li>
</ul>

<p><b>Síntesis.</b> A diferencia de otras sagas donde el protagonismo oscila entre quién tiene más contactos y quién acumula más escenas, en <i>Juego de Tronos</i> <b>Tyrion Lannister</b> es el eje indiscutible de la red global en las cinco métricas analizadas. La evolución por temporadas muestra una transición desde una red centrada en <b>Ned Stark</b> (T1), pasando por una fragmentación regional liderada en peso por <b>Tyrion</b> (T2–T4) y <b>Cersei</b> (T5), hacia un desplazamiento al Norte con <b>Sansa y Jon</b> (T6), hasta culminar en la convergencia de todos los supervivientes en una red altamente densa dominada por el trío <b>Jon, Daenerys y Tyrion</b> (T7–T8).</p>

In [165]:
def serializador_sage(obj):
    if hasattr(obj, "item"):  # Tipos de NumPy (np.int64, np.float64)
        return obj.item()
    try:
        return int(obj) if float(obj).is_integer() else float(obj)
    except (TypeError, ValueError):
        return str(obj)

import os
import math
import json
import numpy as np
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

os.makedirs("grafos_interactivos", exist_ok=True)
os.makedirs("gephi_preconfigurados", exist_ok=True)

PALETA_COMUNIDADES = [
    "#E63946", "#1D3557", "#2A9D8F", "#F4A261",
    "#8338EC", "#06D6A0", "#FF006E", "#3A86FF",
    "#FB5607", "#8AC926", "#6A4C93", "#1982C4",
]
COLOR_RESTO = "#90A4AE"


def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))


def separar_nodos_cercanos(pos, dist_minima=0.065, iteraciones=40):
    """
    Evita que los nodos centrales se monten unos encima de otros aplicando
    una ligera repulsión entre pares que estén por debajo de dist_minima.
    """
    nodos = list(pos.keys())
    coords = np.array([pos[n] for n in nodos], dtype=float)
    n_total = len(nodos)
    if n_total < 2:
        return pos

    for _ in range(iteraciones):
        movido = False
        for i in range(n_total):
            for j in range(i + 1, n_total):
                delta = coords[i] - coords[j]
                dist = np.linalg.norm(delta)
                if dist < dist_minima:
                    movido = True
                    if dist < 1e-6:
                        direccion = np.random.uniform(-1.0, 1.0, size=2)
                        direccion /= np.linalg.norm(direccion)
                    else:
                        direccion = delta / dist
                    empuje = 0.5 * (dist_minima - dist) * direccion
                    coords[i] += empuje
                    coords[j] -= empuje
        if not movido:
            break

    return {nodos[i]: coords[i] for i in range(n_total)}


def calcular_layout_compacto(G, nodo_a_comunidad, n_coms_principales, seed=42):
    """
    Calcula las posiciones dando aire al núcleo central y evitando que las
    componentes aisladas salgan despedidas lejos.
    """
    rng = np.random.RandomState(int(seed))
    componentes = sorted(nx.connected_components(G), key=len, reverse=True)
    
    # 1. Layout de la componente principal con mayor separación
    G_main = G.subgraph(componentes[0]).copy()
    n_main = G_main.number_of_nodes()
    k_optimo = float(4.2 / math.sqrt(n_main)) if n_main < 200 else float(5.5 / math.sqrt(n_main))
    
    pos_init = {}
    for n in G_main.nodes():
        c_idx = nodo_a_comunidad[n] - 1
        if c_idx < n_coms_principales:
            angulo = float(2.0 * math.pi * c_idx / n_coms_principales)
            cx, cy = float(0.75 * math.cos(angulo)), float(0.75 * math.sin(angulo))
        else:
            cx, cy = 0.0, 0.0
        pos_init[n] = np.array([cx + rng.normal(0.0, 0.22), cy + rng.normal(0.0, 0.22)], dtype=float)
        
    pos = nx.spring_layout(
        G_main,
        pos=pos_init,
        k=k_optimo,
        weight="weight",
        iterations=int(300),
        seed=int(seed)
    )
    
    # Normalizar coordenadas de la componente principal al rango [-1, 1]
    coords = np.array(list(pos.values()))
    min_xy = coords.min(axis=0)
    max_xy = coords.max(axis=0)
    span = np.maximum(max_xy - min_xy, 1e-6)
    for n in pos:
        pos[n] = 2.0 * (pos[n] - min_xy) / span - 1.0

    # Despegar nodos que hayan quedado demasiado juntos en el núcleo
    pos = separar_nodos_cercanos(pos, dist_minima=0.065, iteraciones=50)

    # 2. Colocar componentes secundarias (islas) pegadas al borde inferior derecho
    offset_x = 1.25
    offset_y = -0.90
    for comp in componentes[1:]:
        subG = G.subgraph(comp)
        sub_pos = nx.spring_layout(subG, k=0.22, seed=int(seed))
        for n, xy in sub_pos.items():
            pos[n] = np.array([offset_x + float(xy[0]) * 0.10, offset_y + float(xy[1]) * 0.10])
        offset_y += 0.28
        
    return pos


def generar_grafo_interactivo(
    G,
    nombre_archivo,
    titulo,
    metrica_tamano="grado_ponderado",
    seed=42
):
    seed = int(seed)
    G_vis = G.copy()
    n_nodos = int(G_vis.number_of_nodes())
    n_aristas = int(G_vis.number_of_edges())

    # 1. Métricas
    grado = dict(G_vis.degree())
    grado_pond = dict(G_vis.degree(weight="weight"))
    intermediacion = nx.betweenness_centrality(G_vis, weight=None)
    autovector = nx.eigenvector_centrality(G_vis, weight="weight", max_iter=int(1000))

    metricas = {
        "grado": grado,
        "grado_ponderado": grado_pond,
        "intermediacion": intermediacion,
        "autovector": autovector,
    }
    valores_metrica = metricas[metrica_tamano]

    # 2. Comunidades
    comunidades = sorted(
        greedy_modularity_communities(G_vis, weight="weight"),
        key=len,
        reverse=True
    )

    nodo_a_comunidad = {}
    nodo_a_color = {}
    leyenda_items = []

    for idx, com in enumerate(comunidades):
        color = PALETA_COMUNIDADES[idx] if idx < len(PALETA_COMUNIDADES) else COLOR_RESTO
        lider = max(com, key=lambda n: grado_pond[n])
        if idx < 10 and len(com) >= 3:
            leyenda_items.append((color, f"Com. {idx+1}: {lider} ({len(com)} pers.)"))
        for nodo in com:
            nodo_a_comunidad[nodo] = int(idx + 1)
            nodo_a_color[nodo] = color

    # 3. Layout expandido
    pos = calcular_layout_compacto(G_vis, nodo_a_comunidad, min(len(comunidades), 12), seed=seed)

    # 4. Escalado de tamaños, coordenadas y fuentes grandes
    vals = np.array([float(valores_metrica[n]) for n in G_vis.nodes()], dtype=float)
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    escala_lienzo = 1800.0

    nodos_json = []
    for idx_n, n in enumerate(G_vis.nodes()):
        relevancia = float(vals_norm[idx_n])
        tamano = float(20.0 + 54.0 * (relevancia ** 0.75))
        font_size = int(22 + 32 * (relevancia ** 0.7))
        x, y = float(pos[n][0] * escala_lienzo), float(pos[n][1] * escala_lienzo)
        
        tooltip = (
            f"<b>{n}</b> (Comunidad {nodo_a_comunidad[n]})<br>"
            f"Grado: {int(grado[n])}<br>"
            f"Grado ponderado: {int(grado_pond[n])}<br>"
            f"Intermediación: {float(intermediacion[n]):.4f}<br>"
            f"Autovector: {float(autovector[n]):.4f}"
        )
        
        nodos_json.append({
            "id": str(n),
            "label": str(n),
            "x": x,
            "y": y,
            "baseX": x,
            "baseY": y,
            "baseFontSize": font_size,
            "size": tamano,
            "shape": "dot",
            "borderWidth": 5,
            "color": {
                "background": "#B0BEC5",
                "border": nodo_a_color[n],
                "highlight": {"background": "#CFD8DC", "border": nodo_a_color[n]}
            },
            "font": {
                "size": font_size,
                "color": "#111111",
                "face": "Arial",
                "bold": True if relevancia > 0.35 else False,
                "strokeWidth": 5,
                "strokeColor": "#FFFFFF"
            },
            "title": tooltip
        })

    pesos = [float(d.get("weight", 1.0)) for _, _, d in G_vis.edges(data=True)]
    p_max = max(pesos) if pesos else 1.0
    aristas_json = []
    for u, v, d in G_vis.edges(data=True):
        w = float(d.get("weight", 1.0))
        aristas_json.append({
            "from": str(u),
            "to": str(v),
            "value": w,
            "width": float(0.8 + 4.2 * (w / p_max)),
            "title": f"{u} ↔ {v} (Peso: {int(w)})",
            "color": {"color": "rgba(120, 144, 156, 0.28)", "highlight": "#E63946"}
        })

    # 5. Generar archivo HTML interactivo autónomo
    leyenda_html = "".join(
        f'<div style="display:flex;align-items:center;margin-bottom:4px;">'
        f'<span style="width:12px;height:12px;background:{c};display:inline-block;margin-right:6px;border-radius:50%;"></span>'
        f'<span>{lbl}</span></div>'
        for c, lbl in leyenda_items
    )

    html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>{titulo}</title>
  <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
  <style>
    body {{ margin: 0; padding: 0; font-family: Arial, sans-serif; background: #FAFAFA; overflow: hidden; }}
    #header {{
      position: absolute; top: 12px; left: 16px; z-index: 10;
      background: rgba(255,255,255,0.94); padding: 12px 16px; border-radius: 8px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.12); max-width: 480px;
    }}
    #header h2 {{ margin: 0 0 4px 0; font-size: 18px; color: #1A1A1A; }}
    #header p {{ margin: 0; font-size: 12px; color: #555; }}
    #legend {{
      position: absolute; bottom: 16px; left: 16px; z-index: 10;
      background: rgba(255,255,255,0.94); padding: 10px 14px; border-radius: 8px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 12px;
    }}
    #controls {{
      position: absolute; top: 12px; right: 16px; z-index: 10;
      background: rgba(255,255,255,0.94); padding: 10px 12px; border-radius: 8px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.12); font-size: 12px;
      display: flex; flex-direction: column; gap: 8px;
    }}
    .btn-row {{ display: flex; gap: 6px; }}
    .slider-row {{ display: flex; align-items: center; justify-content: space-between; gap: 8px; color: #333; font-weight: bold; }}
    button {{
      padding: 6px 10px; cursor: pointer; border: 1px solid #CCC;
      background: #FFF; border-radius: 4px; font-weight: bold;
    }}
    button:hover {{ background: #EEE; }}
    #mynetwork {{ width: 100vw; height: 100vh; }}
  </style>
</head>
<body>
  <div id="header">
    <h2>{titulo}</h2>
    <p>Nodos: {n_nodos} | Aristas: {n_aristas} | Tamaño: {metrica_tamano} | Borde = Comunidad</p>
  </div>
  <div id="controls">
    <div class="btn-row">
      <button onclick="network.fit({{animation: true}})">Centrar grafo</button>
      <button id="btnPhysics" onclick="togglePhysics()">Activar física</button>
    </div>
    <div class="slider-row">
      <label for="spreadSlider">Separación:</label>
      <input type="range" id="spreadSlider" min="0.5" max="2.5" step="0.1" value="1.0" oninput="escalarDisposicion(this.value)">
      <span id="spreadVal">1.0x</span>
    </div>
    <div class="slider-row">
      <label for="fontSlider">Tamaño texto:</label>
      <input type="range" id="fontSlider" min="0.5" max="2.5" step="0.1" value="1.0" oninput="escalarTexto(this.value)">
      <span id="fontVal">1.0x</span>
    </div>
  </div>
  <div id="legend">
    <div style="font-weight:bold;margin-bottom:6px;">Comunidades (color del borde)</div>
    {leyenda_html}
  </div>
  <div id="mynetwork"></div>

  <script type="text/javascript">
    var nodes = new vis.DataSet({json.dumps(nodos_json, default=serializador_sage)});
    var edges = new vis.DataSet({json.dumps(aristas_json, default=serializador_sage)});
    var container = document.getElementById('mynetwork');
    var data = {{ nodes: nodes, edges: edges }};
    var physicsEnabled = false;

    var options = {{
      nodes: {{
        borderWidth: 5,
        shapeProperties: {{
          useBorderWithImage: true
        }},
        scaling: {{
          label: {{
            enabled: false
          }}
        }}
      }},
      edges: {{
        smooth: {{ type: 'continuous' }}
      }},
      physics: {{
        enabled: false,
        solver: 'forceAtlas2Based',
        forceAtlas2Based: {{
          gravitationalConstant: -180,
          centralGravity: 0.005,
          springLength: 220,
          springConstant: 0.04,
          damping: 0.4,
          avoidOverlap: 1
        }},
        stabilization: {{
          enabled: false
        }}
      }},
      interaction: {{
        hover: true,
        tooltipDelay: 100,
        navigationButtons: true
      }}
    }};

    var network = new vis.Network(container, data, options);
    network.once("afterDrawing", function() {{ network.fit(); }});

    // Carga automática de fotos desde la API de Fandom (si no encuentra, queda el círculo gris)
    async function cargarFotosPersonajes() {{
      var listaNodos = nodes.get();
      for (var i = 0; i < listaNodos.length; i++) {{
        var n = listaNodos[i];
        try {{
          var url = "https://gameofthrones.fandom.com/api.php?action=query&generator=search&gsrsearch=" +
                    encodeURIComponent(n.label) +
                    "&gsrlimit=4&prop=pageimages&piprop=thumbnail&pithumbsize=250&format=json&origin=*";
          var resp = await fetch(url);
          var json = await resp.json();
          if (json.query && json.query.pages) {{
            var paginas = Object.values(json.query.pages).sort(function(a, b) {{ return a.index - b.index; }});
            var personaje = paginas.find(function(p) {{
              var t = (p.title || "").toLowerCase();
              return p.thumbnail && p.thumbnail.source &&
                     !t.includes("season") && !t.includes("episode") &&
                     !t.includes("disambiguation") && !t.includes("category:");
            }});
            if (personaje) {{
              nodes.update({{
                id: n.id,
                shape: "circularImage",
                image: personaje.thumbnail.source
              }});
            }}
          }}
        }} catch (e) {{}}
      }}
    }}
    cargarFotosPersonajes();

    function togglePhysics() {{
      physicsEnabled = !physicsEnabled;
      document.getElementById('btnPhysics').innerText = physicsEnabled ? 'Pausar física' : 'Activar física';
      network.setOptions({{ physics: {{ enabled: physicsEnabled }} }});
    }}

    function escalarDisposicion(factor) {{
      var f = parseFloat(factor);
      document.getElementById('spreadVal').innerText = f.toFixed(1) + 'x';
      var updates = [];
      nodes.forEach(function(n) {{
        updates.push({{
          id: n.id,
          x: n.baseX * f,
          y: n.baseY * f
        }});
      }});
      nodes.update(updates);
      network.fit({{ animation: false }});
    }}

    function escalarTexto(factor) {{
      var f = parseFloat(factor);
      document.getElementById('fontVal').innerText = f.toFixed(1) + 'x';
      var updates = [];
      nodes.forEach(function(n) {{
        updates.push({{
          id: n.id,
          font: {{
            size: Math.round(n.baseFontSize * f)
          }}
        }});
      }});
      nodes.update(updates);
    }}
  </script>
</body>
</html>"""

    ruta_html = f"grafos_interactivos/{nombre_archivo}_{metrica_tamano}.html"
    with open(ruta_html, "w", encoding="utf-8") as f:
        f.write(html_content)

    # 6. Exportar también .gexf para Gephi con las coordenadas expandidas
    for idx_n, n in enumerate(G_vis.nodes()):
        r, g, b = hex_to_rgb(nodo_a_color[n])
        x, y = float(pos[n][0]), float(pos[n][1])
        G_vis.nodes[n]["label"] = str(n)
        G_vis.nodes[n]["grado"] = int(grado[n])
        G_vis.nodes[n]["grado_ponderado"] = float(grado_pond[n])
        G_vis.nodes[n]["intermediacion"] = float(intermediacion[n])
        G_vis.nodes[n]["autovector"] = float(autovector[n])
        G_vis.nodes[n]["comunidad"] = f"Comunidad_{nodo_a_comunidad[n]}"
        G_vis.nodes[n]["viz"] = {
            "color": {"r": int(r), "g": int(g), "b": int(b), "a": 1.0},
            "size": float(15.0 + 60.0 * (vals_norm[idx_n] ** 0.75)),
            "position": {"x": float(x * escala_lienzo), "y": float(y * escala_lienzo), "z": 0.0}
        }

    for u, v, d in G_vis.edges(data=True):
        if "weight" in d:
            d["weight"] = float(d["weight"])

    ruta_gexf = f"gephi_preconfigurados/{nombre_archivo}_{metrica_tamano}.gexf"
    nx.write_gexf(G_vis, ruta_gexf)
    print(f"[OK] Generado: {ruta_html} y {ruta_gexf}")


grafos_estudio = [
    (G_ep1, "01_temporada_1", "Temporada 1 — Estructura de Comunidades e Influencia"),
    (G_ep2, "02_temporada_2", "Temporada 2 — Fragmentación Geográfica (Poniente vs. Essos)"),
    (G_ep3, "03_temporada_3", "Temporada 3 — La Guerra de los Cinco Reyes y la Boda Roja"),
    (G_ep4, "04_temporada_4", "Temporada 4 — La Boda Púrpura y el Juicio de Tyrion"),
    (G_ep5, "05_temporada_5", "Temporada 5 — El Gorrión Supremo y Casa Austera"),
    (G_ep6, "06_temporada_6", "Temporada 6 — El Norte y la Batalla de los Bastardos"),
    (G_ep7, "07_temporada_7", "Temporada 7 — La Llegada de Daenerys y Pozo Dragón"),
    (G_ep8, "08_temporada_8", "Temporada 8 — Convergencia Final y Alta Densidad"),
    (G_general, "00_serie_completa", "Juego de Tronos — Red Global de Interacciones (T1–T8)"),
]

for G_actual, archivo, titulo in grafos_estudio:
    generar_grafo_interactivo(
        G_actual,
        nombre_archivo=archivo,
        titulo=titulo,
        metrica_tamano="grado_ponderado"
    )

# Extra: Red global por intermediación (personajes puente)
generar_grafo_interactivo(
    G_general,
    nombre_archivo="00_serie_completa",
    titulo="Juego de Tronos — Personajes Puente entre Comunidades (T1–T8)",
    metrica_tamano="intermediacion"
)

[OK] Generado: grafos_interactivos/01_temporada_1_grado_ponderado.html y gephi_preconfigurados/01_temporada_1_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/02_temporada_2_grado_ponderado.html y gephi_preconfigurados/02_temporada_2_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/03_temporada_3_grado_ponderado.html y gephi_preconfigurados/03_temporada_3_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/04_temporada_4_grado_ponderado.html y gephi_preconfigurados/04_temporada_4_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/05_temporada_5_grado_ponderado.html y gephi_preconfigurados/05_temporada_5_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/06_temporada_6_grado_ponderado.html y gephi_preconfigurados/06_temporada_6_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/07_temporada_7_grado_ponderado.html y gephi_preconfigurados/07_temporada_7_grado_ponderado.gexf
[OK] Generado: grafos_interactivos/08_temporada_8_grado_ponderado.html y gephi_prec